# NOV-B1: LLM search for new online bin-packing rules (demo)

**What the artifact does.** It runs a FunSearch-style island search that evolves `priority(item, bins)` rules for
online 1-D bin packing. Each new program is written by **Qwen3-8B-FP8** (vLLM 0.11, thinking off) from one or two
parent programs. The FunSearch harness from iteration 9 scores it with a fitness `F`, where 1.0 is best fit and
lower is better. Its *behaviour* is fingerprinted by the bins it picks on 2,000 fixed probe states.

Five sampling arms ran side by side at matched calls: 2,000 per arm and seed, seeds 0 and 1, lock-step rounds of 25.

| arm | what changes |
|---|---|
| `T1.0` | temperature 1.0 |
| `T1.6` | temperature 1.6, top-p 0.95 |
| `DTB` | **decision-token branching** (the proposed operator): force a rank-2..8 token at the highest-entropy code token of the first 48 |
| `RPB` | random-position placebo for DTB (same forcing at a random eligible position) |
| `IDEA` | one appended sentence: write it "based on an idea different from" the parents |

A no-selection arm `NS` (fixed FF/BF parents) ran afterwards. Headline results of the full run:

* **V1 SUPPORTED.** Every core arm's best program beats 0.8 x best fit on both primary held-out sets.
* **V2 PARTIAL.** DTB finds 3.0x the behaviour clusters of T1.0 but only 1.2x those of T1.6 (the bar was 1.5x).
* **V3 SUPPORTED, flagged.** 8 NOVEL rules, all one idea family: prefer a leftover that is a whole multiple of the item size.
* **V6 PARTIAL.** IDEA renames its parents less often than T1.0, not more.

**What this notebook runs (CPU only, a few minutes).** The artifact's entry point `method.py` runs ten stages as
subprocesses on a GPU machine. Here every CPU stage runs in-process with the original code:

* the instance sets and their hashes, and the harness gate that reproduces FunSearch's printed numbers;
* the probe states;
* one round of the island search for all five arms;
* behaviour clusters, the V1, V3 and V6 rules, and a behaviour-space map.

The one thing a CPU cannot run, the 8B model, is **replayed**: its recorded completions are served in place of vLLM.
Everything around them is recomputed and checked against the recorded run: the prompts (rendered with the real
Qwen3 tokenizer), the island database, the per-call vLLM seeds, sandboxed evaluation, fitness, probe decisions and
the DTB/RPB fork choices.

## 1. Setup

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru: NOT on Colab, always install (every original script logs with it)
_pip('loguru==0.7.3')

# Pre-installed on Colab, install locally only, at Colab's versions: numpy, scipy, matplotlib;
# transformers, tokenizers, huggingface_hub and jinja2 (Qwen3 tokenizer and chat template); aiohttp (the vLLM client)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'transformers==5.0.0', 'tokenizers==0.22.2',
         'huggingface_hub==1.4.0', 'jinja2==3.1.6', 'aiohttp==3.14.5')

In [ ]:
from __future__ import annotations

import argparse
import ast
import asyncio
import hashlib
import json
import keyword
import math
import os
import pickle
import re
import shutil
import signal
import subprocess
import sys
import time
import types
import warnings
from collections import Counter
from concurrent.futures import Future
from pathlib import Path
from typing import Callable

# sandbox.py sets these before numpy is imported (one BLAS thread per evaluation)
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import aiohttp  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402  (notebook: figures)
import numpy as np  # noqa: E402
from loguru import logger  # noqa: E402
from scipy.cluster.hierarchy import fcluster, linkage  # noqa: E402
from scipy.sparse import csr_matrix  # noqa: E402
from scipy.sparse.csgraph import connected_components  # noqa: E402

# the original scripts' logger set-up (method.py main): INFO lines to stdout
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")


def as_module(name: str, names: str) -> types.SimpleNamespace:
    """Notebook helper. The original code is split over modules (bp.py, sandbox.py, engine.py, ...) that call each
    other as `bp.pack(...)`, `sandbox.compile_program(...)`, `engine.sample(...)`. Here each module's code runs in the
    notebook namespace; this bundles the names a module defines under the module's name, so those calls stay as written."""
    g = globals()
    return types.SimpleNamespace(__name__=name, **{n: g[n] for n in names.split()})

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_jSassd-YWLVC/round-10/experiment-51/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
EXAMPLES = data["datasets"][0]["examples"]
print(data["metadata"]["description"])
print(f"{len(EXAMPLES)} examples by role:", dict(Counter(r for e in EXAMPLES for r in e["metadata_role"])))
print("held-out instance sets shipped:", {k: f"{len(v)} x {v[0]['num_items']} items, capacity {v[0]['capacity']}"
                                          for k, v in data["metadata"]["heldout_instances"].items()})

`mini_demo_data.json` holds one dataset, `novb1_search_programs` (the search-call rows of the run), curated to 96 examples:

* **`replay_round0`, 85 rows.** Seed 0, round 0, calls 0-16 of each of the five arms. Round 0 is the window a partial
  dump can replay faithfully: every island still holds only the seed program (best fit), so the recorded completions
  answer exactly the prompts the scheduler draws.
* **`best_of_arm`, 5 rows.** The V1 best program of each arm, found at rounds 49-78 of the full run.
* **`novel`, 8 rows.** The programs the arm-blind manual read classed NOVEL (two of them are also arm bests).

Every row keeps what the replay and its checks need: the code, the recorded fitness, the 25 bin counts, the DTB/RPB
fork record, the completion token ids, and the sha256 of the recorded probe-decision vector. The two primary
held-out instance sets and the run's headline numbers ride along in `metadata`.

## 2. Configuration

Every tunable value lives here, with the original value in the comment. The shipped values use everything
`mini_demo_data.json` supports: all five arms x 17 replayed calls and all 11 notable programs. They also use the
full original 3,843-point ab grid. Smaller values, such as 1 arm, 1 call and 1 notable program with
`AB_A_STEP, AB_B_STEP = 10, 30`, run every cell in under a minute.

**Replay mode** (the default) can serve only round 0 of seed 0, calls 0-16. A full run means 5 arms x seeds 0, 1 x
80 rounds x 25 calls with `USE_LIVE_ENGINE = True`. That needs a vLLM server with Qwen/Qwen3-8B-FP8 on a GPU
(`start_vllm.sh` in the artifact). It also needs hours, not minutes.

* `N_NOTABLE` is how many of the 11 notable programs to re-score.
* `AB_A_STEP` and `AB_B_STEP` thin the 3,843-point ab-heuristic grid that the novelty screen compares against.
* `ROOT` is the working folder that stands in for the artifact folder: data, results, probes and search rows are
  written there.

In [ ]:
# ---- island search (method.py stage "search")
SEARCH_ARMS = ["T1.0", "T1.6", "DTB", "RPB", "IDEA"]   # original: the same five arms
SEARCH_SEEDS = [0]          # original: [0, 1]
ROUNDS = 1                  # original: 80 (replay mode: 1, only round 0 is shipped)
CALLS_PER_ROUND = 17        # original: 25 (replay mode: at most 17)
USE_LIVE_ENGINE = False     # original: True, i.e. a vLLM server with Qwen/Qwen3-8B-FP8 (start_vllm.sh, GPU)
VLLM_URL = "http://127.0.0.1:8051"
# ---- notable programs (the V1 best of each arm + the 8 NOVEL programs) to re-score on selection and held-out sets
N_NOTABLE = 11              # original: all 11 (V1 used the 5 arm bests; V3 audited 30 representatives)
# ---- ab-heuristic grid of the known-heuristic corpus (a = 0..20, b = 0..60, base FF/BF/WF)
AB_A_STEP, AB_B_STEP = 1, 1     # original: 1, 1 (3,843 points)

ROOT = Path("novb1_demo").resolve()   # notebook: stands in for the artifact folder (Path(__file__).parent...)
for sub in ("data/instances", "data/probes", "data/corpus", "results", "logs"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
assert USE_LIVE_ENGINE or (ROUNDS == 1 and CALLS_PER_ROUND <= 17 and SEARCH_SEEDS == [0]), \
    "replay mode serves round 0 of seed 0, calls 0..16 only (set USE_LIVE_ENGINE = True with a vLLM server for more)"

## 3. `method.py`: the pipeline map

This is the artifact's entry point. Each stage is a resumable script under `src/`, run as a subprocess, and writes
`results/*.json`. The code below is `method.py` verbatim apart from `ROOT`. It needs the full repository, a GPU and API
keys, so the notebook only prints its stage table. The sections after it run each CPU stage's code in-process instead:

| stage | original script(s) | in this notebook |
|---|---|---|
| `s0` | `s0_verify.py`, `gate.py` | §4: instance sets and their sha256, then the G-HARNESS gate |
| `s1` | `probes.py`, `corpus.py`, `unit_tests.py` | §4: the P1 probe states; §6: the rebuildable part of the known-heuristic corpus |
| `vllm`, `smoke`, `search` | `start_vllm.sh`, `search.py` | §5: one lock-step round of all five arms, with the LLM replayed |
| `ns` | `search.py --arms NS` | not run (reference numbers only) |
| `post` | `postsearch.py` | §6: behaviour clusters, V2 rates, V6, V1, V3 |
| `judges`, `v5` | `judges.py` (OpenRouter), `v5.py` (vLLM logprobs) | not run (reference numbers only) |
| `final` | `v3_final.py`, `figures.py`, ... | §7: tables and figures |

In [ ]:
# ---- method.py (verbatim; ROOT is the config cell's folder)
PY = [sys.executable, "-I"]
ENV = dict(os.environ, OMP_NUM_THREADS="1", HF_HUB_OFFLINE=os.environ.get("HF_HUB_OFFLINE", "1"))
STAGES = {
    "s0": [["src/s0_verify.py", "verify"], ["src/gate.py"], ["src/s0_verify.py", "gate"]],
    "s1": [["src/probes.py"], ["src/corpus.py", "10"], ["tests/unit_tests.py"]],
    "smoke": [["src/search.py", "--run", "smoke", "--seeds", "99", "--rounds", "1", "--calls-per-round", "10"]],
    "search": [["src/search.py", "--run", "main", "--seeds", "0,1", "--rounds", "80", "--pause-after", "3"]],
    "ns": [["src/search.py", "--run", "ns", "--seeds", "0,1", "--rounds", "40", "--arms", "NS"]],
    "post": [["src/postsearch.py", "clusters,best,v12,v3"]],
    "judges": [["src/judges.py", "gate"], ["src/judges.py", "candidates"]],
    "v5": [["src/v5.py"]],
    "final": [["src/v3_final.py"], ["src/postsearch.py", "v6,evaluator,placebo"], ["src/figures.py"],
              ["src/costs.py"], ["audit/rederive.py"], ["src/outputs.py"]],
}


def run(stage: str) -> None:
    if stage == "vllm":
        pid = subprocess.run(["bash", str(ROOT / "start_vllm.sh")], capture_output=True, text=True).stdout.strip()
        logger.info(f"vLLM server PID {pid} (poll logs/vllm.log for 'GPU KV cache size' and /health)")
        return
    for cmd in STAGES[stage]:
        t0 = time.time()
        logger.info(f"[{stage}] {' '.join(cmd)}")
        r = subprocess.run(PY + [str(ROOT / cmd[0])] + cmd[1:], cwd=ROOT, env=ENV)
        if r.returncode != 0:
            raise RuntimeError(f"stage {stage} failed at {cmd} (exit {r.returncode})")
        logger.info(f"[{stage}] done in {time.time() - t0:.0f}s")


@logger.catch(reraise=True)
def main() -> None:
    logger.remove()
    logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
    logger.add(ROOT / "logs" / "method.log", rotation="30 MB", level="DEBUG")
    stages = sys.argv[1:] or ["final"]
    for st in stages:
        run(st)


# notebook: main() is not called here (it needs the repository's src/ scripts); this is the order it runs them in
for st in ["s0", "s1", "vllm", "smoke", "search", "ns", "post", "judges", "v5", "final"]:
    print(f"{st:7s}", "bash start_vllm.sh" if st == "vllm" else " ; ".join(" ".join(c) for c in STAGES[st]))

## 4. Stages s0 and s1: harness, instance sets, gate, probe states

### 4.1 `bp.py`: the exact FunSearch online bin-packing harness

The semantics are those of google-deepmind/funsearch `bin_packing.ipynb`:

* One bin is pre-allocated per item, all at full capacity.
* `priority(item, bins[valid])` scores every bin the item fits in, unused bins included.
* The item goes to the arg-max (first index on ties).
* The score is the number of bins whose remaining capacity is no longer the full capacity.

In [ ]:
# ---- bp.py (ROOT comes from the config cell; originally Path(__file__).resolve().parent.parent)
INST_DIR = ROOT / "data" / "instances"
EXT = ROOT / "ext"


class ShapeError(Exception):
    """Priority output is not a 1-D array with one score per valid bin."""


# ----------------------------------------------------------------------------------------------- harness
def valid(item: int, bins: np.ndarray) -> np.ndarray:
    return np.nonzero((bins - item) >= 0)[0]


def pack(items: list[int], cap: int, prio: Callable) -> int:
    """Pack `items` online with FunSearch semantics; return the number of used bins."""
    bins = np.array([cap for _ in range(len(items))])
    for it in items:
        v = np.nonzero(bins >= it)[0]  # identical to (bins - item) >= 0 for integer data
        p = np.asarray(prio(it, bins[v]))
        if p.ndim != 1 or p.shape[0] != v.shape[0]:
            raise ShapeError(f"priority returned shape {p.shape} for {v.shape[0]} valid bins")
        b = v[np.argmax(p)]
        bins[b] -= it
    return int((bins != cap).sum())


def pack_trace(items: list[int], cap: int, prio: Callable) -> tuple[int, list[tuple[int, list[int], int, int]]]:
    """Pack and record states: (item, remaining caps of opened bins that fit in index order, n_empty, chosen cap)."""
    n = len(items)
    bins = np.array([cap for _ in range(n)])
    used = np.zeros(n, dtype=bool)
    states = []
    for it in items:
        v = np.nonzero(bins >= it)[0]
        vb = bins[v]
        p = np.asarray(prio(it, vb))
        b = v[np.argmax(p)]
        opened_fit = [int(x) for x, u in zip(vb, used[v]) if u]
        n_empty = int((~used).sum())
        states.append((int(it), opened_fit, n_empty, int(bins[b])))
        bins[b] -= it
        used[b] = True
    return int(used.sum()), states


def l1(items: list[int], cap: int) -> int:
    return int(math.ceil(sum(items) / cap))


def family_excess(bins: list[int] | np.ndarray, l1s: list[int] | np.ndarray) -> float:
    """FunSearch's excess: (mean bins - mean L1) / mean L1 over a family of instances."""
    mb, ml = float(np.mean(bins)), float(np.mean(l1s))
    return (mb - ml) / ml

**Instance sets.** The selection, calibration, probe-source and generated held-out sets come from seeded generators.
The selection sets are the 25 instances F is computed on: 20 uniform U[20, 100] instances at capacity 150 with 120
items, and 5 Weibull(3, 45) instances at capacity 100 with 5,000 items. The external held-out sets are the FunSearch
notebook's Weibull 5k and OR-Library's binpack1-4. `build_all_sets()` reads them from `ext/`; the notebook takes the
two primary ones (`H-FSW5k`, `H-OR3`) from `mini_demo_data.json` instead.

In [ ]:
# ----------------------------------------------------------------------------------------------- instances
def weibull_items(rng: np.random.Generator, n: int, shape: float = 3.0, scale: float = 45.0, hi: int = 100) -> list[int]:
    x = np.clip(np.round(rng.weibull(shape, size=n) * scale), 1, hi).astype(int)
    return [int(v) for v in x]


def uniform_items(rng: np.random.Generator, n: int, lo: int = 20, hi: int = 100) -> list[int]:
    return [int(v) for v in rng.integers(lo, hi + 1, size=n)]


def _inst(name: str, cap: int, items: list[int]) -> dict:
    return {"id": name, "capacity": int(cap), "num_items": len(items), "items": items, "l1": l1(items, cap)}


def gen_sets() -> dict[str, list[dict]]:
    """All generated instance sets (seeds fixed in the plan)."""
    s: dict[str, list[dict]] = {}
    s["SEL-OR"] = [_inst(f"selor_{i}", 150, uniform_items(np.random.default_rng(91001 + i), 120)) for i in range(20)]
    s["SEL-W"] = [_inst(f"selw_{i}", 100, weibull_items(np.random.default_rng(92001 + i), 5000)) for i in range(5)]
    s["H-W5k"] = [_inst(f"hw5k_{i}", 100, weibull_items(np.random.default_rng(93001 + i), 5000)) for i in range(5)]
    s["H-W10k"] = [_inst(f"hw10k_{i}", 100, weibull_items(np.random.default_rng(94001 + i), 10000)) for i in range(5)]
    s["H-SHIFT"] = [_inst(f"hshift_{i}", 150, weibull_items(np.random.default_rng(97001 + i), 5000, 7.0, 75.0, 150))
                    for i in range(5)]
    s["CALIB-W"] = [_inst(f"calw_{i}", 100, weibull_items(np.random.default_rng(99001 + i), 5000)) for i in range(20)]
    s["P1SRC-W"] = [_inst(f"p1w_{i}", 100, weibull_items(np.random.default_rng(95001 + i), 5000)) for i in range(5)]
    s["P1SRC-OR"] = [_inst(f"p1or_{i}", 150, uniform_items(np.random.default_rng(96001 + i), 500)) for i in range(5)]
    return s


def parse_orlib(path: Path) -> list[dict]:
    tok = path.read_text().split()
    p = int(tok[0])
    out, k = [], 1
    for _ in range(p):
        name = tok[k]
        cap, n, best = int(float(tok[k + 1])), int(tok[k + 2]), int(tok[k + 3])
        items = [int(float(x)) for x in tok[k + 4:k + 4 + n]]
        k += 4 + n
        d = _inst(name, cap, items)
        d["best_known"] = best
        out.append(d)
    return out


def funsearch_notebook() -> tuple[dict, dict, str, str]:
    """Return (OR3 dict, Weibull 5k dict, OR heuristic source, Weibull heuristic source) parsed as data."""
    nb = json.loads((EXT / "funsearch" / "bin_packing.ipynb").read_text())
    src = "".join(nb["cells"][2]["source"])
    tree = ast.parse(src)
    data = {}
    for node in tree.body:
        if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Subscript):
            key = ast.literal_eval(node.targets[0].slice)
            data[key] = ast.literal_eval(node.value)
    or_src = "".join(nb["cells"][8]["source"]).split("\n\n\n# Test performance")[0]
    w_src = "".join(nb["cells"][9]["source"]).split("\n\n\n# Test performance")[0]
    return data["OR3"], data["Weibull 5k"], or_src, w_src


def nb_dict_to_insts(d: dict) -> list[dict]:
    return [_inst(k, v["capacity"], [int(x) for x in v["items"]]) for k, v in d.items()]


def sha_insts(insts: list[dict]) -> str:
    h = hashlib.sha256()
    for x in insts:
        h.update(json.dumps([x["id"], x["capacity"], x["items"]]).encode())
    return h.hexdigest()


def build_all_sets() -> dict[str, list[dict]]:
    sets = gen_sets()
    for i in range(1, 5):
        sets[f"H-OR{i}"] = parse_orlib(EXT / "orlib" / f"binpack{i}.txt")
    or3, w5k, _, _ = funsearch_notebook()
    sets["FS-OR3"] = nb_dict_to_insts(or3)
    sets["H-FSW5k"] = nb_dict_to_insts(w5k)
    return sets


def save_sets(sets: dict[str, list[dict]]) -> dict[str, str]:
    INST_DIR.mkdir(parents=True, exist_ok=True)
    hashes = {}
    for name, insts in sets.items():
        (INST_DIR / f"{name}.json").write_text(json.dumps(insts))
        hashes[name] = sha_insts(insts)
    return hashes


def load_set(name: str) -> list[dict]:
    return json.loads((INST_DIR / f"{name}.json").read_text())

**Heuristics and fitness.** The reference rules are First, Best and Worst Fit, FunSearch's two published heuristics
(`c12` for OR and `c14` for Weibull), and the ab family of Herrmann & Pallez (arXiv 2510.27353). The selection
fitness is `F = 0.5 * exc_OR / exc_OR(BF) + 0.5 * exc_W / exc_W(BF)` over the 25 selection instances, so best fit
scores exactly 1.0.

In [ ]:
# ----------------------------------------------------------------------------------------------- heuristics
def prio_ff(item, bins):
    return -np.arange(len(bins))


def prio_bf(item, bins):
    return -(bins - item)


def prio_wf(item, bins):
    score = (bins - item).astype(float)
    m = bins == bins.max()
    if not m.all():
        score[m] = -1e9
    return score


def prio_c12_vec(item, bins):
    """Vectorised exact equivalent of FunSearch's OR heuristic (c12)."""
    g = bins - item
    return np.select([g <= 2, g <= 3, g <= 5, g <= 7, g <= 9, g <= 12, g <= 15, g <= 18, g <= 20, g <= 21],
                     [4, 3, 2, 1, 0.9, 0.95, 0.97, 0.98, 0.98, 0.98], 0.99)


def prio_c14(item, bins):
    max_bin_cap = max(bins)
    score = (bins - max_bin_cap) ** 2 / item + bins ** 2 / (item ** 2)
    score += bins ** 2 / item ** 3
    score[bins > item] = -score[bins > item]
    score[1:] -= score[:-1]
    return score


def make_ab(a: int, b: int, base: str) -> Callable:
    """ab-heuristics of Herrmann & Pallez (arXiv 2510.27353, Algorithms 4-6); capacity = bins.max() (an unused bin
    at full capacity is always among the valid bins in the pre-allocated harness)."""
    def prio(item, bins):
        cap = bins.max()
        with np.errstate(divide="ignore", invalid="ignore"):
            first = cap - bins + 1
            if base == "FF":
                return np.where(bins <= item + a, first, np.where(bins < item + b, -2, 1))
            if base == "BF":
                return np.where(bins <= item + a, first, np.where(bins < item + b, -2.0, 1.0 / (bins - item)))
            return np.where(bins <= item + a, first, np.where(bins <= item + b, -2.0,
                                                              np.where(bins == cap, -1.0, -1.0 / (bins - item))))
    prio.__name__ = f"ab{base}_{a}_{b}"
    return prio


def ab_source(a: int, b: int, base: str) -> str:
    """Program text of an ab member in the search's program format (for probes/corpus)."""
    if base == "FF":
        tail = "np.where(bins < item + b, -2, 1)"
    elif base == "BF":
        tail = "np.where(bins < item + b, -2.0, 1.0 / (bins - item))"
    else:
        tail = "np.where(bins <= item + b, -2.0, np.where(bins == cap, -1.0, -1.0 / (bins - item)))"
    return (f"def priority(item, bins):\n    a, b = {a}, {b}\n    cap = bins.max()\n"
            f"    with np.errstate(divide='ignore', invalid='ignore'):\n"
            f"        return np.where(bins <= item + a, cap - bins + 1, {tail})\n")


SRC_FF = "def priority(item, bins):\n    return -np.arange(len(bins))\n"
SRC_BF = "def priority(item, bins):\n    return -(bins - item)\n"
SRC_WF = ("def priority(item, bins):\n    score = (bins - item).astype(float)\n    m = bins == bins.max()\n"
          "    if not m.all():\n        score[m] = -1e9\n    return score\n")


def harmonic_k(items: list[int], cap: int, k: int) -> int:
    """Lee & Lee Harmonic-k (standalone, stateful): class j<k items (cap/(j+1) < s <= cap/j) packed j per bin;
    class-k items (s <= cap/k) packed by Next Fit in their own bins."""
    count = {j: 0 for j in range(1, k)}
    bins_used = 0
    nf_rem = None
    for s in items:
        j = next((j for j in range(1, k) if s * (j + 1) > cap and s * j <= cap), k)
        if j < k:
            if count[j] == 0:
                bins_used += 1
            count[j] = (count[j] + 1) % j
        else:
            if nf_rem is None or nf_rem < s:
                bins_used += 1
                nf_rem = cap
            nf_rem -= s
    return bins_used


# ----------------------------------------------------------------------------------------------- fitness
class Fitness:
    """F = 0.5*exc_OR/exc_OR(BF) + 0.5*exc_W/exc_W(BF) on the 25 selection instances (lower is better, F(BF)=1)."""

    def __init__(self, sel_or: list[dict], sel_w: list[dict], bf_or_exc: float, bf_w_exc: float):
        self.l1_or = [x["l1"] for x in sel_or]
        self.l1_w = [x["l1"] for x in sel_w]
        self.n_or = len(sel_or)
        self.bf_or_exc, self.bf_w_exc = bf_or_exc, bf_w_exc

    def parts(self, bins25: list[int]) -> tuple[float, float]:
        return family_excess(bins25[:self.n_or], self.l1_or), family_excess(bins25[self.n_or:], self.l1_w)

    def F(self, bins25: list[int]) -> float:
        eo, ew = self.parts(bins25)
        return 0.5 * eo / self.bf_or_exc + 0.5 * ew / self.bf_w_exc


def eval_set(insts: list[dict], prio: Callable) -> list[int]:
    return [pack(x["items"], x["capacity"], prio) for x in insts]


bp = as_module("bp", "ROOT INST_DIR EXT ShapeError valid pack pack_trace l1 family_excess weibull_items uniform_items "
                     "_inst gen_sets parse_orlib funsearch_notebook nb_dict_to_insts sha_insts build_all_sets save_sets "
                     "load_set prio_ff prio_bf prio_wf prio_c12_vec prio_c14 make_ab ab_source SRC_FF SRC_BF SRC_WF "
                     "harmonic_k Fitness eval_set")

### 4.2 `sandbox.py`: running LLM-written programs safely

Every candidate program goes through the same checks before it runs:

* an AST screen (only `numpy` and `math` imports, no dunders, no `open`/`exec`/`eval`, no numpy file I/O);
* a restricted builtins dict (`print` is a no-op);
* `SIGPROF` CPU-time limits per instance set and for the probes.

The original runs tasks in N persistent *spawn* workers, each capped by `RLIMIT_AS`, with a parent watchdog that
kills and respawns hung workers. A notebook kernel cannot host spawn workers whose code lives in cells, so
**`EvalPool` is the one structural substitution.** Each task runs in-process through the body of the original
worker loop, with the same CPU-time limits. The memory and file-size caps are not applied, since they would cap the
whole kernel. `submit()` still returns a `Future`, so every caller is unchanged.

In [ ]:
# ---- sandbox.py (screen, restricted exec, normalised keys)
ALLOWED_MODULES = {"numpy", "math"}
FORBIDDEN_NAMES = {"open", "exec", "eval", "compile", "__import__", "globals", "locals", "vars", "getattr", "setattr",
                   "delattr", "input", "breakpoint"}
# extra I/O layer (not in the plan's list; documented): numpy file/ctypes entry points
FORBIDDEN_ATTRS = {"save", "savez", "savez_compressed", "savetxt", "load", "loadtxt", "fromfile", "tofile", "memmap",
                   "genfromtxt", "fromregex", "ctypeslib", "ctypes", "DataSource", "f2py", "distutils", "testing"}
ERR_CODE = 65535
ERROR_TYPES = ("ok", "syntax", "forbidden", "runtime", "shape", "timeout", "memory")


class Forbidden(Exception):
    pass


class CpuTimeout(Exception):
    pass


def screen(tree: ast.AST) -> None:
    """Reject imports other than numpy/math, forbidden names and any dunder attribute or name."""
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for a in node.names:
                if a.name not in ALLOWED_MODULES:
                    raise Forbidden(f"import {a.name}")
        elif isinstance(node, ast.ImportFrom):
            if node.module not in ALLOWED_MODULES or node.level:
                raise Forbidden(f"from {node.module} import")
        elif isinstance(node, ast.Name):
            if node.id in FORBIDDEN_NAMES or (node.id.startswith("__") and node.id.endswith("__")):
                raise Forbidden(f"name {node.id}")
        elif isinstance(node, ast.Attribute):
            if node.attr.startswith("__") or node.attr in FORBIDDEN_ATTRS:
                raise Forbidden(f"attribute {node.attr}")


def _restricted_import(name, globals=None, locals=None, fromlist=(), level=0):  # noqa: A002
    if level or name.split(".")[0] not in ALLOWED_MODULES:
        raise ImportError(f"import of {name} not allowed")
    return __import__(name, globals, locals, fromlist, level)


def _noop(*_a, **_k):
    return None


_SAFE = ["abs", "all", "any", "bool", "dict", "divmod", "enumerate", "filter", "float", "frozenset", "int",
         "isinstance", "iter", "len", "list", "map", "max", "min", "next", "pow", "range", "reversed", "round", "set",
         "slice", "sorted", "str", "sum", "tuple", "zip", "complex", "hash", "Exception", "ValueError", "TypeError",
         "ZeroDivisionError", "IndexError", "KeyError", "ArithmeticError", "OverflowError", "FloatingPointError",
         "RuntimeError", "StopIteration", "AssertionError", "NotImplementedError", "True", "False", "None"]


def safe_builtins() -> dict:
    import builtins
    d = {k: getattr(builtins, k) for k in _SAFE if hasattr(builtins, k)}
    d["print"] = _noop
    d["__import__"] = _restricted_import
    return d


def normalise_key(code: str) -> str | None:
    """sha256 of ast.dump of the program with docstrings removed (comments are dropped by ast)."""
    try:
        tree = ast.parse(code)
    except (SyntaxError, ValueError):
        return None
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.body:
            b0 = node.body[0]
            if isinstance(b0, ast.Expr) and isinstance(getattr(b0, "value", None), ast.Constant) \
                    and isinstance(b0.value.value, str):
                node.body = node.body[1:] or [ast.Pass()]
    return hashlib.sha256(ast.dump(tree, annotate_fields=False, include_attributes=False).encode()).hexdigest()


def compile_program(code: str):
    """Return (priority function, error_type, message)."""
    try:
        tree = ast.parse(code)
    except (SyntaxError, ValueError) as e:  # TabError/IndentationError are SyntaxError subclasses
        return None, "syntax", repr(e)[:300]
    try:
        screen(tree)
    except Forbidden as e:
        return None, "forbidden", str(e)[:300]
    ns = {"np": np, "math": math, "__builtins__": safe_builtins()}
    try:
        exec(compile(tree, "<program>", "exec"), ns)  # noqa: S102 - screened, restricted namespace, rlimited worker
    except SyntaxError as e:
        return None, "syntax", repr(e)[:300]
    except MemoryError:
        return None, "memory", "MemoryError at definition"
    except Exception as e:  # noqa: BLE001
        return None, "runtime", repr(e)[:300]
    fn = ns.get("priority")
    if not callable(fn):
        return None, "syntax", "no function named priority"
    return fn, "ok", ""

In [ ]:
# ---- sandbox.py (worker side) + the in-process EvalPool
def _on_sigprof(signum, frame):
    raise CpuTimeout("cpu time limit")


def _classify(e: BaseException) -> str:
    # notebook: `from bp import ShapeError` dropped; ShapeError is defined in the bp cell
    if isinstance(e, CpuTimeout):
        return "timeout"
    if isinstance(e, MemoryError):
        return "memory"
    if isinstance(e, ShapeError):
        return "shape"
    return "runtime"


def _probe_decisions(fn, states: list[tuple[int, np.ndarray]]) -> np.ndarray:
    out = np.full(len(states), ERR_CODE, dtype=np.uint16)
    for i, (item, arr) in enumerate(states):
        try:
            p = np.asarray(fn(item, arr.copy()))
            if p.ndim != 1 or p.shape[0] != arr.shape[0]:
                continue
            out[i] = int(arr[int(np.argmax(p))])
        except CpuTimeout:
            raise
        except MemoryError:
            raise
        except Exception:  # noqa: BLE001 - per-state errors are recorded as ERR
            continue
    return out


def _load_probe(path: Path) -> list[tuple[int, np.ndarray]]:
    import json
    if not path.exists():
        return []
    d = json.loads(path.read_text())
    return [(int(it), np.array(list(of) + [int(cap)] * int(ne))) for it, of, ne, cap in d["states"]]


class EvalPool:
    """Notebook stand-in for sandbox.EvalPool (N persistent spawn workers, RLIMIT_AS / RLIMIT_FSIZE caps, parent
    watchdog). Each task runs in-process through the body of the original _worker_main loop, with the same SIGPROF
    CPU-time limits; submit() returns an already-resolved Future, so every caller is unchanged."""

    def __init__(self, n_workers: int, root: Path, as_gib: float = 2.0, log=None):
        self.root, self.as_bytes, self.log = str(root), int(as_gib * 1024 ** 3), log
        self.n_respawn = 0
        self._tid = 0
        # _worker_main's set-up, applied to the kernel (RLIMIT_AS / RLIMIT_FSIZE would cap the whole kernel: skipped)
        warnings.simplefilter("ignore")
        np.seterr(all="ignore")
        signal.signal(signal.SIGPROF, _on_sigprof)
        self.sets_cache: dict[str, list[dict]] = {}
        self.probes = {"P1": _load_probe(Path(root) / "data" / "probes" / "P1.json"),
                       "P2": _load_probe(Path(root) / "data" / "probes" / "P2.json")}

    def get_set(self, name):
        if name not in self.sets_cache:
            self.sets_cache[name] = bp.load_set(name)
        return self.sets_cache[name]

    def _run(self, task: dict) -> dict:
        # ---- body of the original _worker_main loop for one task (`conn.send(res)` becomes `return res`)
        get_set, probes = self.get_set, self.probes
        res = {"tid": task["tid"], "error_type": "ok", "msg": "", "bins": {}, "eval_s": 0.0, "probes": {}}
        t0 = time.process_time()
        fn, et, msg = compile_program(task["code"])
        if fn is None:
            res.update(error_type=et, msg=msg)
            return res
        # ---- evaluation on instance sets (fail fast, in the given order)
        for set_name in task.get("sets", []):
            insts = get_set(set_name)
            try:
                signal.setitimer(signal.ITIMER_PROF, float(task["limit"]))
                ts = time.process_time()
                res["bins"][set_name] = [bp.pack(x["items"], x["capacity"], fn) for x in insts]
                res.setdefault("set_s", {})[set_name] = time.process_time() - ts
            except BaseException as e:  # noqa: BLE001 - classify every failure of untrusted code
                signal.setitimer(signal.ITIMER_PROF, 0)
                if isinstance(e, KeyboardInterrupt):
                    raise
                res["error_type"] = _classify(e)
                res["msg"] = f"{set_name}: {e!r}"[:300]
                res["failed_set"] = set_name
                if task.get("fail_fast", True):
                    break
            finally:
                signal.setitimer(signal.ITIMER_PROF, 0)
        res["eval_s"] = time.process_time() - t0
        # ---- behaviour probes (only for programs valid on all requested sets)
        if res["error_type"] == "ok" and task.get("probes"):
            for pname in task["probes"]:
                try:
                    signal.setitimer(signal.ITIMER_PROF, float(task.get("probe_limit", 20.0)))
                    res["probes"][pname] = _probe_decisions(fn, probes[pname]).tobytes()
                except BaseException as e:  # noqa: BLE001
                    if isinstance(e, KeyboardInterrupt):
                        raise
                    res["probes"][pname] = None
                    res["probe_error"] = f"{pname}: {e!r}"[:200]
                finally:
                    signal.setitimer(signal.ITIMER_PROF, 0)
        return res

    def submit(self, task: dict) -> Future:
        self._tid += 1
        task = dict(task, tid=self._tid)
        fut: Future = Future()
        fut.set_result(self._run(task))
        return fut

    def backlog(self) -> int:
        return 0

    def close(self) -> None:
        signal.setitimer(signal.ITIMER_PROF, 0)


sandbox = as_module("sandbox", "ALLOWED_MODULES FORBIDDEN_NAMES FORBIDDEN_ATTRS ERR_CODE ERROR_TYPES Forbidden CpuTimeout "
                               "screen safe_builtins normalise_key compile_program _classify _probe_decisions "
                               "_load_probe EvalPool")

### 4.3 Instance sets and the S0 hash check

`s0_verify.py verify` requires every instance set to hash to the sha256 embedded in the pre-registered plan. The
seeded generators reproduce all eight generated sets bit for bit. The two shipped held-out sets are checked against
the same table. binpack3 hashes identically to the FunSearch notebook's OR3, so it also serves as `FS-OR3`.

In [ ]:
# ---- s0_verify.py: the sha256 of every instance set as embedded in the plan
EMBEDDED = {
    "SEL-OR": "4fa84e1e8d2fcb7d7a49f2fc4b6dff3c14c6980f741d7e6c9804c46f47de82d7",
    "SEL-W": "7f95af860eabf585434030cbb95b62f217a7b23443bbed0e60f1b8c2974e68d3",
    "H-W5k": "eb4d3c6629da9ac1533985963a22edede519dd73c75f19d51c0190ff3c505b1a",
    "H-W10k": "cf5cd9b599a58b5932602759e844e112fd08cb04cb38536bc9767ebcb7f8f4f0",
    "H-SHIFT": "6a98e8c395aeb14fb47cf24f334dfd84d575ee320391ec76c670ff361d7ba9ff",
    "CALIB-W": "5e7b0468448900469cdf8e8756188bfce30b4ce3e3f6c43277f3d3bb72127cdd",
    "P1SRC-W": "e6eb2fb86a9da2d6fa849099b25cc4712061858f7af18d41ca4f78e5bc3af6e1",
    "P1SRC-OR": "3a2a73f5af220e978e00614ea087769306ed0da0f18a440f90502c150fdc715c",
    "H-OR1": "c5604b1a056c7b1070b58846550e1b991bf3e47eac59167d5886641957b358a6",
    "H-OR2": "0e1fd208f26139b94359b4abb0ca2a1574f70b7cceda139fc30ac8612307d13c",
    "H-OR3": "b89390f503db5bd471361efb3b9aa57c147be39678f78f64d1fe13df533d31a1",
    "FS-OR3": "b89390f503db5bd471361efb3b9aa57c147be39678f78f64d1fe13df533d31a1",
    "H-OR4": "1014d8c716c511745b902a5bbd82db66b7fbcc1541267aabd2a477927299c2d5",
    "H-FSW5k": "e58e2ff43f97b0093407a2efdbd68b0dce316b79c5352e39073c53ce57ee9703",
}

# notebook: bp.build_all_sets() = gen_sets() + four OR-Library files + the FunSearch notebook (under ext/, not shipped);
# here gen_sets() + the two primary held-out sets from mini_demo_data.json (H-OR1/2/4 are not used below)
sets = bp.gen_sets()
for name, insts in data["metadata"]["heldout_instances"].items():
    sets[name] = insts
sets["FS-OR3"] = sets["H-OR3"]
hashes = bp.save_sets(sets)
for name, h in hashes.items():
    print(f"  {name:9s} {len(sets[name]):2d} instances x {sets[name][0]['num_items']:5d} items  sha256 {h[:16]}...  "
          f"{'== plan' if h == EMBEDDED[name] else '!= plan ' + EMBEDDED[name][:16]}")
print("S0 instance hashes:", "ALL MATCH" if all(hashes[n] == EMBEDDED[n] for n in hashes) else "MISMATCH")

### 4.4 `gate.py`: G-HARNESS, reproducing FunSearch's printed numbers

No search call is allowed unless the harness reproduces the numbers FunSearch printed:

* best fit: 212.0 bins (5.37 %) on OR3, and 2067.0 bins (3.98 %) on Weibull 5k;
* FunSearch's own heuristics: 207.45 bins (3.11 %) for c12 on OR3, and 2001.4 bins (0.68 %) for c14 on Weibull 5k.

The gate also checks two more things. Best fit on 20 fresh Weibull instances must stay within 0.25 points of 3.98 %.
The priority forms of FF/BF/WF must equal hand-coded packers. It writes `results/harness_gate.json`, whose best-fit
bin counts seed the search.

In [ ]:
# ---- gate.py
def hand_ff(items, cap):
    rem = []
    for s in items:
        for i, r in enumerate(rem):
            if r >= s:
                rem[i] -= s
                break
        else:
            rem.append(cap - s)
    return len(rem)


def hand_bf(items, cap):
    rem = []
    for s in items:
        best, bi = None, -1
        for i, r in enumerate(rem):
            if r >= s and (best is None or r < best):
                best, bi = r, i
        if bi < 0:
            rem.append(cap - s)
        else:
            rem[bi] -= s
    return len(rem)


def hand_wf(items, cap):
    rem = []
    for s in items:
        best, bi = None, -1
        for i, r in enumerate(rem):
            if r >= s and (best is None or r > best):
                best, bi = r, i
        if bi < 0:
            rem.append(cap - s)
        else:
            rem[bi] -= s
    return len(rem)


# ---- body of gate.py main() (sets / hashes come from the cell above)
out: dict = {"utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
out["instance_sha256"] = hashes
out["instance_counts"] = {k: [len(v), int(np.mean([x["num_items"] for x in v]))] for k, v in sets.items()}
# binpack3.txt must equal the notebook's OR3 item for item
same = all(a["id"].strip() == b["id"].strip() and a["items"] == b["items"] and a["capacity"] == b["capacity"]
           for a, b in zip(sets["H-OR3"], sets["FS-OR3"])) and len(sets["H-OR3"]) == len(sets["FS-OR3"])
out["binpack3_equals_notebook_OR3"] = bool(same)
logger.info(f"binpack3 == notebook OR3: {same}")

# notebook: `_, _, or_src, w_src = bp.funsearch_notebook()` compiled the notebook's verbatim c12 / c14 through the
# sandbox; the notebook file is not shipped, so the bp function forms stand in (the original run verified
# prio_c12_vec == c12 per instance, and prio_c14 is FunSearch's Weibull heuristic)
c12, c14 = bp.prio_c12_vec, bp.prio_c14


def fam(name, prio):
    insts = sets[name]
    t = time.perf_counter()
    b = bp.eval_set(insts, prio)
    dt = time.perf_counter() - t
    mb, ml = float(np.mean(b)), float(np.mean([x["l1"] for x in insts]))
    return {"mean_bins": mb, "mean_l1": ml, "excess_pct": 100 * (mb - ml) / ml, "bins": b, "secs": dt}


np.seterr(all="ignore")
g = {}
g["BF_OR3"] = fam("FS-OR3", bp.prio_bf)
g["BF_binpack3"] = fam("H-OR3", bp.prio_bf)
g["BF_W5k"] = fam("H-FSW5k", bp.prio_bf)
g["c12vec_OR3"] = fam("FS-OR3", bp.prio_c12_vec)
g["c14_W5k"] = fam("H-FSW5k", c14)
targets = {"BF_OR3": (212.0, 5.37), "BF_binpack3": (212.0, 5.37), "BF_W5k": (2067.0, 3.98),
           "c12vec_OR3": (207.45, 3.11), "c14_W5k": (2001.4, 0.68)}
ok = True
for k, (mb, ex) in targets.items():
    passed = abs(g[k]["mean_bins"] - mb) <= 0.01 and abs(round(g[k]["excess_pct"], 2) - ex) < 1e-9
    g[k]["target"] = [mb, ex]
    g[k]["pass"] = bool(passed)
    ok &= passed
    logger.info(f"{k}: mean={g[k]['mean_bins']:.3f} excess={g[k]['excess_pct']:.2f}% target={mb}/{ex}% "
                f"{'PASS' if passed else 'FAIL'} ({g[k]['secs']:.1f}s)")
out["gate"] = g
out["G_HARNESS_PASS"] = bool(ok and same)

# calibration of the Weibull generator
cal = fam("CALIB-W", bp.prio_bf)
out["calibration"] = {"bf_excess_pct": cal["excess_pct"], "target": 3.98, "tol": 0.25,
                      "pass": abs(cal["excess_pct"] - 3.98) <= 0.25}
logger.info(f"calibration BF excess on 20 fresh 5k: {cal['excess_pct']:.3f}% pass={out['calibration']['pass']}")

# BF reference on selection sets and timing
bf_or = fam("SEL-OR", bp.prio_bf)
bf_w = fam("SEL-W", bp.prio_bf)
out["sel_bf"] = {"or_excess": bf_or["excess_pct"] / 100, "w_excess": bf_w["excess_pct"] / 100,
                 "or_bins": bf_or["bins"], "w_bins": bf_w["bins"], "secs": bf_or["secs"] + bf_w["secs"]}
out["sel_limit_s"] = max(10.0, 8 * out["sel_bf"]["secs"])
logger.info(f"SEL BF excess OR={bf_or['excess_pct']:.3f}% W={bf_w['excess_pct']:.3f}% t={out['sel_bf']['secs']:.2f}s")

# hand-coded packer equivalence (3 instances each from SEL-OR and SEL-W)
eq = {}
for nm, prio, hand in [("FF", bp.prio_ff, hand_ff), ("BF", bp.prio_bf, hand_bf), ("WF", bp.prio_wf, hand_wf)]:
    res = []
    for x in sets["SEL-OR"][:3] + sets["SEL-W"][:1]:
        res.append(bp.pack(x["items"], x["capacity"], prio) == hand(x["items"], x["capacity"]))
    eq[nm] = all(res)
out["priority_vs_handcoded"] = eq
logger.info(f"priority-form vs hand-coded packers: {eq}")
(ROOT / "results" / "harness_gate.json").write_text(json.dumps(out, indent=1))
logger.info(f"G-HARNESS {'PASS' if out['G_HARNESS_PASS'] else 'FAIL'}")

### 4.5 `probes.py`: the P1 behaviour probes

Behaviour is measured on 2,000 fixed states, all drawn from `default_rng(20261024)`:

* 1,000 states drawn from the packing traces of six reference heuristics (BF, FF, WF, AWF, c12, c14) on 10 instances;
* 1,000 uniform random states.

A program's *decision* on a state is the remaining capacity of the bin it picks. Two programs behave alike when
they agree on at least 95 % of the states. The construction is deterministic, so the file's sha256 must match the
one the original run committed.

In [ ]:
# ---- probes.py (ROOT from the config cell)
PROBE_DIR = ROOT / "data" / "probes"
ERR = 65535
SEED = 20261024

SRC_AWF = ("def priority(item, bins):\n    cap = bins.max()\n    opn = np.nonzero(bins < cap)[0]\n"
           "    out = np.zeros(len(bins), dtype=float)\n    if len(opn) >= 2:\n"
           "        order = np.argsort(-bins[opn], kind='stable')\n        out[opn[order[1]]] = 1.0\n"
           "    elif len(opn) == 1:\n        out[opn[0]] = 1.0\n    else:\n"
           "        out[np.nonzero(bins == cap)[0][0]] = 1.0\n    return out\n")


def prio_awf(item, bins):
    """Almost Worst Fit (Johnson 1974): the open bin with the second-largest remaining capacity (stable sort,
    descending); the only open bin if there is one; otherwise the first empty bin. One-hot float vector."""
    cap = bins.max()
    opn = np.nonzero(bins < cap)[0]
    out = np.zeros(len(bins), dtype=float)
    if len(opn) >= 2:
        order = np.argsort(-bins[opn], kind="stable")
        out[opn[order[1]]] = 1.0
    elif len(opn) == 1:
        out[opn[0]] = 1.0
    else:
        out[np.nonzero(bins == cap)[0][0]] = 1.0
    return out


REFS = [("BF", bp.prio_bf), ("FF", bp.prio_ff), ("WF", bp.prio_wf), ("AWF", prio_awf),
        ("c12", bp.prio_c12_vec), ("c14", bp.prio_c14)]


def trace_at(items: list[int], cap: int, prio, steps: set[int]) -> dict[int, tuple[int, list[int], int, int]]:
    """bp.pack_trace semantics, recording only the requested steps (memory: the full trace holds ~10^8 ints)."""
    n = len(items)
    bins = np.array([cap for _ in range(n)])
    used = np.zeros(n, dtype=bool)
    out = {}
    for t, it in enumerate(items):
        v = np.nonzero(bins >= it)[0]
        vb = bins[v]
        p = np.asarray(prio(it, vb))
        b = v[np.argmax(p)]
        if t in steps:
            opened_fit = [int(x) for x, u in zip(vb, used[v]) if u]
            out[t] = (int(it), opened_fit, int((~used).sum()), int(bins[b]))
        bins[b] -= it
        used[b] = True
    return out


def decide(prio, item: int, arr: np.ndarray) -> int:
    """Exactly sandbox._probe_decisions for one state (trusted reference code)."""
    try:
        p = np.asarray(prio(item, arr.copy()))
        if p.ndim != 1 or p.shape[0] != arr.shape[0]:
            return ERR
        return int(arr[int(np.argmax(p))])
    except Exception:  # noqa: BLE001 - per-state errors are ERR, as in the sandbox
        return ERR


def state_arrays(states: list) -> list[tuple[int, np.ndarray]]:
    return [(int(it), np.array(list(of) + [int(cap)] * int(ne))) for it, of, ne, cap in states]


def agreement(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a), np.asarray(b)
    return float(np.mean((a == b) & (a != ERR)))


def build() -> dict:
    t0 = time.time()
    insts = bp.load_set("P1SRC-W")[:5] + bp.load_set("P1SRC-OR")[:5]
    sizes = [len(x["items"]) for x in insts]
    n_total = len(REFS) * sum(sizes)
    rng = np.random.default_rng(SEED)
    idx = rng.choice(n_total, 1000, replace=False)  # kept in draw order
    uniform = []
    for j in range(1000):
        cap = 100 if j < 500 else 150
        item = int(rng.integers(1, cap))
        n_open = int(rng.integers(1, 31))
        open_fit = [int(v) for v in rng.integers(item, cap, size=n_open).tolist()]
        n_empty = int(rng.integers(1, 31))
        uniform.append([item, open_fit, n_empty, cap])
    # locate every drawn global index: trace order = for ref, for inst, for step
    where = {}
    g = 0
    for ri, _ in enumerate(REFS):
        for ii, n in enumerate(sizes):
            where[(ri, ii)] = (g, g + n)
            g += n
    assert g == n_total
    need: dict[tuple[int, int], set[int]] = {}
    loc = []
    for gi in idx.tolist():
        for key, (lo, hi) in where.items():
            if lo <= gi < hi:
                need.setdefault(key, set()).add(gi - lo)
                loc.append((key, gi - lo))
                break
    recorded = {}
    for (ri, ii), steps in sorted(need.items()):
        rname, prio = REFS[ri]
        tr = trace_at(insts[ii]["items"], insts[ii]["capacity"], prio, steps)
        for s, st in tr.items():
            recorded[(ri, ii, s)] = st
    trace_states, src, chosen = [], [], []
    for (ri, ii), s in loc:
        it, of, ne, ch = recorded[(ri, ii, s)]
        trace_states.append([it, of, ne, int(insts[ii]["capacity"])])
        src.append([REFS[ri][0], insts[ii]["id"], s])
        chosen.append(ch)
    states = trace_states + uniform
    meta = {"construction": "plan NOV-B1 S1 PROBES; rng default_rng(20261024); choice(165000, 1000, replace=False) "
                            "kept in draw order, then 1,000 uniform states from the same rng",
            "refs": [r for r, _ in REFS], "insts": [x["id"] for x in insts], "n_trace_total": n_total,
            "trace_src": src, "trace_recorded_choice": chosen,
            "uniform_departure": "item = rng.integers(1, cap) i.e. 1..cap-1 so that open_fit = integers(item, cap) is non-empty",
            "format": "[item, open_fit, n_empty, capacity]"}
    PROBE_DIR.mkdir(parents=True, exist_ok=True)
    blob = json.dumps({"states": states, "meta": meta}, separators=(",", ":"))
    (PROBE_DIR / "P1.json").write_text(blob)
    file_sha = hashlib.sha256(blob.encode()).hexdigest()
    states_sha = hashlib.sha256(json.dumps(states, separators=(",", ":")).encode()).hexdigest()
    # ---- self-check: prefix-opening references reproduce their own recorded choices on their own trace states
    arrs = state_arrays(states)
    self_agree = {}
    for ri, (rname, prio) in enumerate(REFS):
        ks = [k for k in range(1000) if src[k][0] == rname]
        hits = sum(decide(prio, arrs[k][0], arrs[k][1]) == chosen[k] for k in ks)
        self_agree[rname] = {"n_states": len(ks), "agree": hits / max(1, len(ks))}
    # reference decision vectors on all 2,000 states
    ref_dec = {rname: [decide(prio, it, a) for it, a in arrs] for rname, prio in REFS}
    ref_agree = {f"{a}~{b}": agreement(ref_dec[a], ref_dec[b]) for i, (a, _) in enumerate(REFS)
                 for (b, _) in REFS[i + 1:]}
    res = {"utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "file": "data/probes/P1.json",
           "file_sha256": file_sha, "states_sha256": states_sha, "n_states": len(states),
           "n_trace": len(trace_states), "n_uniform": len(uniform),
           "trace_states_per_ref": {r: sum(1 for x in src if x[0] == r) for r, _ in REFS},
           "self_agreement": self_agree, "reference_pair_agreement": ref_agree,
           "self_check_pass": all(v["agree"] == 1.0 for r, v in self_agree.items() if r != "c14"),
           "c14_note": "c14's in-place score[1:] -= score[:-1] is index-dependent and can open a later empty bin, "
                       "so its rebuilt arrays are not exact; its self-agreement is reported, not required",
           "secs": time.time() - t0}
    return res


def tests(res: dict) -> dict:
    """Testing plan item 3."""
    d = json.loads((PROBE_DIR / "P1.json").read_text())
    arrs = state_arrays(d["states"])
    bf = np.array([decide(bp.prio_bf, it, a) for it, a in arrs])
    # notebook: `import sandbox` dropped (the sandbox namespace is defined above)
    renamed = ("def priority(item, bins):\n    # best fit, renamed and reformatted\n    leftover = bins - item\n"
               "    return -1.0 * leftover\n")
    fn, et, _ = sandbox.compile_program(renamed)
    assert fn is not None, et
    ren = sandbox._probe_decisions(fn, arrs)
    wf = np.array([decide(bp.prio_wf, it, a) for it, a in arrs])
    err = np.full(len(arrs), ERR, dtype=np.uint16)
    out = {"renamed_bf_vs_bf": agreement(ren, bf), "bf_vs_wf": agreement(bf, wf), "err_vs_err": agreement(err, err),
           "state_format_capacity_last": all(s[3] in (100, 150) and len(s) == 4 for s in d["states"])}
    out["pass"] = (out["renamed_bf_vs_bf"] == 1.0 and out["bf_vs_wf"] < 0.95 and out["err_vs_err"] == 0.0
                   and out["state_format_capacity_last"])
    return out


probes = as_module("probes", "PROBE_DIR ERR SEED SRC_AWF prio_awf REFS trace_at decide state_arrays agreement build tests")

# probes.py main() builds twice (rebuild must be byte-identical); notebook: build once and compare with the committed sha
r1 = build()
r1["tests"] = tests(r1)
(ROOT / "results" / "probes.json").write_text(json.dumps(r1, indent=1))
ref_pr = data["metadata"]["reference"]["probes"]
print(f"P1: {r1['n_states']} states ({r1['n_trace']} trace + {r1['n_uniform']} uniform) in {r1['secs']:.1f}s")
print(f"  file sha256   {r1['file_sha256'][:16]}...  {'== committed' if r1['file_sha256'] == ref_pr['file_sha256'] else '!= committed'}")
print(f"  states sha256 {r1['states_sha256'][:16]}...  {'== committed' if r1['states_sha256'] == ref_pr['states_sha256'] else '!= committed'}")
print(f"  self-check pass: {r1['self_check_pass']}  tests: {r1['tests']}")
print("  reference-pair agreement:", {k: round(v, 3) for k, v in list(r1["reference_pair_agreement"].items())[:6]}, "...")

## 5. Stage `search`: the island search, with the LLM replayed

### 5.1 `islands.py`: FunSearch's programs database

The database follows FunSearch's semantics, scaled down:

* 5 islands per (seed, arm).
* Programs cluster on their signature, the 25 per-instance bin counts.
* A prompt draws 2 clusters by a softmax over their scores (`-100 F`), using a temperature schedule.
* Within a cluster, shorter programs are preferred.
* Parents appear in the prompt worse first.
* Every 400 calls, the worst 2 islands are reset from a surviving island's best program.

In [ ]:
# ---- islands.py
N_ISLANDS, FUNCS_PER_PROMPT, T_INIT, T_PERIOD, RESET_EVERY = 5, 2, 0.1, 30000, 400
SCORE_SCALE = -100.0


def softmax(logits: np.ndarray, temperature: float) -> np.ndarray:
    """FunSearch's _softmax: scipy-style softmax of logits / temperature, renormalised at the argmax."""
    x = np.asarray(logits, dtype=np.float64) / temperature
    x = x - x.max()
    p = np.exp(x)
    p /= p.sum()
    i = int(np.argmax(p))
    p[i] = 1.0 - (p[:i].sum() + p[i + 1:].sum())
    return p


class Cluster:
    def __init__(self, score: float, prog: dict):
        self.score = score
        self.programs = [prog]
        self.lengths = [len(prog["code"])]

    def register(self, prog: dict) -> None:
        self.programs.append(prog)
        self.lengths.append(len(prog["code"]))

    def sample(self, rng: np.random.Generator) -> dict:
        L = np.array(self.lengths, dtype=np.float64)
        p = softmax(-(L - L.min()) / (L.max() + 1e-6), 1.0)
        return self.programs[int(rng.choice(len(self.programs), p=p))]


class Island:
    def __init__(self):
        self.clusters: dict[tuple, Cluster] = {}
        self.n_programs = 0

    def register(self, prog: dict, score: float, sig: tuple) -> None:
        if sig in self.clusters:
            self.clusters[sig].register(prog)
        else:
            self.clusters[sig] = Cluster(score, prog)
        self.n_programs += 1

    def get_prompt_parents(self, rng: np.random.Generator) -> list[dict]:
        sigs = list(self.clusters.keys())
        scores = np.array([self.clusters[s].score for s in sigs])
        temp = T_INIT * (1 - (self.n_programs % T_PERIOD) / T_PERIOD)
        p = softmax(scores, temp)
        k = min(len(self.clusters), FUNCS_PER_PROMPT)
        idx = rng.choice(len(sigs), size=k, p=p)  # with replacement, as FunSearch
        chosen = [(self.clusters[sigs[i]].sample(rng), self.clusters[sigs[i]].score) for i in idx]
        order = np.argsort([s for _, s in chosen], kind="stable")  # worse first
        return [chosen[i][0] for i in order]


class Database:
    """One programs database per (seed, arm)."""

    def __init__(self, seed: int, arm_idx: int, seed_prog: dict, seed_score: float, seed_sig: tuple):
        self.rng_parents = np.random.default_rng([seed, arm_idx, 1])
        self.rng_reset = np.random.default_rng([seed, arm_idx, 2])
        self.islands = [Island() for _ in range(N_ISLANDS)]
        self.best_score = [-np.inf] * N_ISLANDS
        self.best_prog: list[dict | None] = [None] * N_ISLANDS
        self.best_sig: list[tuple | None] = [None] * N_ISLANDS
        for i in range(N_ISLANDS):
            self.register(seed_prog, seed_score, seed_sig, i)
        self.n_resets = 0
        self.log: list[dict] = []

    def register(self, prog: dict, score: float, sig: tuple, island: int) -> None:
        self.islands[island].register(prog, score, sig)
        if score > self.best_score[island]:
            self.best_score[island] = score
            self.best_prog[island] = prog
            self.best_sig[island] = sig

    def draw(self) -> tuple[int, list[dict]]:
        isl = int(self.rng_parents.integers(0, N_ISLANDS))
        return isl, self.islands[isl].get_prompt_parents(self.rng_parents)

    def reset(self) -> dict:
        noisy = np.array(self.best_score) + self.rng_reset.standard_normal(N_ISLANDS) * 1e-6
        order = np.argsort(noisy, kind="stable")
        n_reset = N_ISLANDS // 2
        reset_ids, keep_ids = order[:n_reset], order[n_reset:]
        ev = {"reset": [int(x) for x in reset_ids], "founders": []}
        for isl in reset_ids:
            isl = int(isl)
            founder = int(self.rng_reset.choice(keep_ids))
            prog, score, sig = self.best_prog[founder], self.best_score[founder], self.best_sig[founder]
            self.islands[isl] = Island()
            self.best_score[isl] = -np.inf
            self.best_prog[isl] = None
            self.register(prog, score, sig, isl)
            ev["founders"].append([isl, founder, prog["pid"], float(score)])
        self.n_resets += 1
        self.log.append(ev)
        return ev

    def n_clusters(self) -> list[int]:
        return [len(i.clusters) for i in self.islands]


islands = as_module("islands", "N_ISLANDS FUNCS_PER_PROMPT T_INIT T_PERIOD RESET_EVERY SCORE_SCALE softmax Cluster "
                               "Island Database")

### 5.2 `prompts.py`: what the model sees

The prompt has three parts:

* a short task preamble;
* the parents rendered as `priority_v0` and `priority_v1`, with FunSearch's docstrings and worse parent first;
* the instruction "write `priority_vk`".

The assistant turn is prefilled with the next signature, so the model writes only a function body. The prompt is
identical across arms, except for the one sentence IDEA appends. The cell prints the round-0 prompt, whose only
parent is best fit.

In [ ]:
# ---- prompts.py
PREAMBLE = ("Online 1-D bin packing: items arrive one at a time and each must be placed immediately. priority(item, bins) "
            "receives the item size and a numpy array with the remaining capacities of every bin that can hold the item "
            "(unused bins, at full capacity, included); the item goes to the bin with the highest score. The goal is to "
            "use as few bins as possible. Only numpy (np) and math may be used.")
MODULE_DOC = '"""Finds good assignment for online 1d bin packing."""'
V0_DOC = ('    """Returns priority with which we want to add item to each bin.\n\n'
          '    Args:\n'
          '        item: Size of item to be added to the bin.\n'
          '        bins: Array of capacities for each bin.\n\n'
          '    Return:\n'
          '        Array of same size as bins with priority score of each bin.\n'
          '    """\n')
SIG = "def priority_v{i}(item: float, bins: np.ndarray) -> np.ndarray:\n"
STOPS = ["```", "\ndef ", "\nclass ", "\nif __name__", "\nprint("]
MAX_TOKENS = 512
SEED_BODY_BF = "    return -(bins - item)\n"
SEED_BODY_FF = "    return -np.arange(len(bins))\n"


def body_of(program: str) -> str:
    """Body of a stored program (everything after the first line 'def priority(item, bins):')."""
    first, _, rest = program.partition("\n")
    assert first.startswith("def priority("), first
    return rest


def render_version(body: str, i: int) -> str:
    body = re.sub(r"\bpriority\s*\(", f"priority_v{i}(", body)
    doc = V0_DOC if i == 0 else f'    """Improved version of `priority_v{i - 1}`."""\n'
    return SIG.format(i=i) + doc + body.rstrip("\n") + "\n"


def user_message(bodies: list[str], idea: bool) -> str:
    """bodies are sorted worse first (v0 = worst). k = len(bodies) in {1, 2}."""
    k = len(bodies)
    block = MODULE_DOC + "\nimport numpy as np\n\n\n" + "\n\n".join(render_version(b, i) for i, b in enumerate(bodies))
    instr = f"Write priority_v{k}, an improved version of priority_v{k - 1}. Return only the code of priority_v{k}."
    if idea:
        instr += (f" Write priority_v{k} based on an idea different from "
                  f"{'the function above' if k == 1 else 'both functions above'}.")
    return PREAMBLE + "\n\n```python\n" + block + "```\n\n" + instr


def prefill(k: int) -> str:
    return "```python\n" + SIG.format(i=k) + f'    """Improved version of `priority_v{k - 1}`."""\n'


def render(tokenizer, bodies: list[str], idea: bool) -> tuple[str, list[int]]:
    """Chat template (thinking off, generation prompt) + assistant prefill; tokenised once."""
    msgs = [{"role": "user", "content": user_message(bodies, idea)}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    text += prefill(len(bodies))
    ids = tokenizer.encode(text, add_special_tokens=False)
    return text, ids


def trim_body(completion: str, k: int) -> str:
    """Keep lines up to the first non-empty line with indent 0; rename recursive calls priority_v{k}( -> priority(."""
    out = []
    for ln in completion.split("\n"):
        if ln.strip() and not ln[0].isspace():
            break
        out.append(ln)
    body = "\n".join(out).rstrip() + "\n"
    if not body.strip():
        body = ""
    return re.sub(rf"\bpriority_v{k}\s*\(", "priority(", body)


def program_from(completion: str, k: int) -> str:
    return "def priority(item, bins):\n" + trim_body(completion, k)


prompts = as_module("prompts", "PREAMBLE MODULE_DOC V0_DOC SIG STOPS MAX_TOKENS SEED_BODY_BF SEED_BODY_FF body_of "
                               "render_version user_message prefill render trim_body program_from")

# notebook: the round-0 user message (every island holds only the seed program, best fit) and IDEA's extra sentence
msg_round0 = prompts.user_message([prompts.body_of(bp.SRC_BF)], idea=False)
print(msg_round0)
print("\nIDEA appends:", prompts.user_message([prompts.body_of(bp.SRC_BF)], idea=True)[len(msg_round0):])
print("\nassistant prefill:\n" + prompts.prefill(1))

### 5.3 `engine.py`: the five sampling arms and decision-token branching

`T1.0`, `IDEA` and `NS` sample at temperature 1.0, and `T1.6` at temperature 1.6 with top-p 0.95. `DTB` and `RPB`
make two requests:

1. **REQ1** samples 48 tokens at temperature 1.0 and returns the top-20 log-probabilities of each.
2. `choose_fork` picks a position `j` among the eligible *code* tokens: not inside a comment or string, not
   whitespace, and with at least one allowed candidate at ranks 2-8. **DTB** takes the highest top-20 entropy;
   **RPB** takes a uniform random eligible position.
3. A token is drawn uniformly from the rank-2..8 candidates at `j` and forced.
4. **REQ2** continues from `prompt + gen1[:j] + [forced]`.

Per-call vLLM seeds are a sha256 hash of `(seed, arm index, call id)`, and the forced-token draws use
`default_rng([seed, arm_idx, call_id, 7])`. Both can be re-derived below without a GPU.

In [ ]:
# ---- engine.py (`import prompts` dropped: the prompts namespace is defined above)
ENGINE_TAG = "vllm0.11.0|Qwen3-8B-FP8@220b46e|fp8-marlin-w8a16"
ARMS = ["T1.0", "T1.6", "DTB", "RPB", "IDEA", "NS"]
ARM_IDX = {a: i for i, a in enumerate(ARMS)}
SAMPLING = {"T1.0": (1.0, 1.0), "T1.6": (1.6, 0.95), "DTB": (1.0, 1.0), "RPB": (1.0, 1.0), "IDEA": (1.0, 1.0),
            "NS": (1.0, 1.0)}
REQ1_TOKENS, N_LOGPROBS, RANK_LO, RANK_HI = 48, 20, 2, 8
HTTP_TIMEOUT_S, HTTP_RETRIES = 180, 2


def call_seed(*parts: int) -> int:
    h = hashlib.sha256(("|".join(str(p) for p in parts)).encode()).digest()
    return int.from_bytes(h[:4], "little") & 0x7FFFFFFF


class Engine:
    def __init__(self, base_url: str, tokenizer, max_inflight: int = 48, model: str = "qwen3-8b"):
        self.url = base_url.rstrip("/") + "/v1/completions"
        self.tok, self.model = tokenizer, model
        self.sem = asyncio.Semaphore(max_inflight)
        self.session: aiohttp.ClientSession | None = None
        self.special = {tokenizer.eos_token_id, tokenizer.pad_token_id,
                        tokenizer.convert_tokens_to_ids("<|im_end|>"),
                        tokenizer.convert_tokens_to_ids("<|endoftext|>")} - {None}
        self._tok_text: dict[int, str] = {}
        self.n_http_fail = 0

    async def __aenter__(self):
        self.session = aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=HTTP_TIMEOUT_S))
        return self

    async def __aexit__(self, *exc):
        await self.session.close()

    def ttext(self, tid: int) -> str:
        if tid not in self._tok_text:
            self._tok_text[tid] = self.tok.decode([tid])
        return self._tok_text[tid]

    async def complete(self, prompt_ids: list[int], *, max_tokens: int, temperature: float, top_p: float, seed: int,
                       logprobs: int) -> dict:
        payload = {"model": self.model, "prompt": prompt_ids, "max_tokens": max_tokens, "temperature": temperature,
                   "top_p": top_p, "seed": seed, "stop": prompts.STOPS, "logprobs": logprobs,
                   "return_tokens_as_token_ids": True, "skip_special_tokens": False}
        last = ""
        for attempt in range(HTTP_RETRIES + 1):
            try:
                async with self.sem:
                    async with self.session.post(self.url, json=payload) as r:
                        if r.status != 200:
                            last = f"HTTP {r.status}: {(await r.text())[:300]}"
                            continue
                        d = await r.json()
                ch = d["choices"][0]
                lp = ch.get("logprobs") or {}
                ids = [int(t.split(":", 1)[1]) for t in lp.get("tokens") or []]
                tops = []
                for tl in lp.get("top_logprobs") or []:
                    tops.append(sorted(((int(k.split(":", 1)[1]), float(v)) for k, v in (tl or {}).items()),
                                       key=lambda x: -x[1]))
                return {"ok": True, "text": ch["text"], "ids": ids, "tops": tops,
                        "finish_reason": ch.get("finish_reason"), "stop_reason": ch.get("stop_reason"),
                        "n_tokens": int(d.get("usage", {}).get("completion_tokens", len(ids))), "attempts": attempt + 1}
            except (aiohttp.ClientError, asyncio.TimeoutError, KeyError, ValueError) as e:
                last = repr(e)[:300]
        self.n_http_fail += 1
        return {"ok": False, "error": last, "text": "", "ids": [], "tops": [], "finish_reason": None, "n_tokens": 0}


# ------------------------------------------------------------------------------------------ DTB / RPB helpers
def scan_inside(text: str) -> list[bool]:
    """Per character: inside a '#' comment or a string literal (opening/closing quotes and '#' count as inside).
    The prefill docstring is already closed when the body starts."""
    inside = [False] * len(text)
    i, n = 0, len(text)
    while i < n:
        c = text[i]
        if c == "#":
            while i < n and text[i] != "\n":
                inside[i] = True
                i += 1
            continue
        if c in "'\"":
            q3 = text[i:i + 3]
            if q3 in ("'''", '"""'):
                j = i + 3
                while j < n and text[j:j + 3] != q3:
                    j += 2 if text[j] == "\\" else 1
                end = min(n, j + 3)
            else:
                j = i + 1
                while j < n and text[j] != c and text[j] != "\n":
                    j += 2 if text[j] == "\\" else 1
                end = min(n, j + 1)
            for k in range(i, end):
                inside[k] = True
            i = end
            continue
        i += 1
    return inside


def body_char_limit(text: str) -> int:
    """Characters of the decoded completion that survive trimming (up to the first non-empty indent-0 line)."""
    pos = 0
    for ln in text.split("\n"):
        if ln.strip() and not ln[0].isspace():
            return pos
        pos += len(ln) + 1
    return len(text)


def token_class(tok_text: str, prev_text: str) -> str:
    s = tok_text.strip()
    if not s:
        return "whitespace"
    if s in ("np", "math") or (prev_text.rstrip().endswith("np.") and re.fullmatch(r"[A-Za-z_]\w*", s)):
        return "np-call"
    if keyword.iskeyword(s) or s in ("True", "False", "None"):
        return "keyword"
    if re.fullmatch(r"[A-Za-z_]\w*", s):
        return "identifier"
    if re.fullmatch(r"\.?\d[\d_.eE+-]*", s):
        return "number"
    if re.fullmatch(r"[+\-*/%<>=!&|^~@]+", s):
        return "operator"
    return "punctuation"


def entropy_top(top: list[tuple[int, float]]) -> float:
    lp = np.array([v for _, v in top[:N_LOGPROBS]], dtype=float)
    if lp.size == 0:
        return 0.0
    p = np.exp(lp - lp.max())
    p /= p.sum()
    return float(-(p * np.log(np.clip(p, 1e-300, None))).sum())


def allowed(eng: Engine, tid: int) -> bool:
    if tid in eng.special:
        return False
    t = eng.ttext(tid)
    if not t or not t.strip() or "```" in t:
        return False
    return re.search(r"\n\S", t) is None


def choose_fork(eng: Engine, gen1: dict, mode: str, rng: np.random.Generator) -> dict:
    """Return the fork decision for DTB ('entropy') or RPB ('random') on REQ1's output."""
    ids, tops = gen1["ids"], gen1["tops"]
    texts = [eng.ttext(t) for t in ids]
    full = "".join(texts)
    inside = scan_inside(full)
    limit = body_char_limit(gen1["text"])
    H, elig, starts, pos = [], [], [], 0
    for t, (tid, tt) in enumerate(zip(ids, texts)):
        starts.append(pos)
        H.append(entropy_top(tops[t]) if t < len(tops) else 0.0)
        span = range(pos, pos + len(tt))
        code = all(not inside[c] for c in span if not full[c].isspace())
        ws = not tt.strip()
        cands = [(r + 1, cid) for r, (cid, _) in enumerate(tops[t][:N_LOGPROBS]) if RANK_LO <= r + 1 <= RANK_HI
                 and allowed(eng, cid)] if t < len(tops) else []
        if code and not ws and cands and pos < limit:
            elig.append((t, cands))
        pos += len(tt)
    info = {"H": [round(h, 4) for h in H], "n_E": len(elig), "E": [t for t, _ in elig]}
    if len(elig) < 2:
        info["unforced"] = True
        return info
    if mode == "entropy":
        j_idx = int(np.argmax([H[t] for t, _ in elig]))  # earliest on ties
    else:
        j_idx = int(rng.integers(0, len(elig)))
    j, cands = elig[j_idx]
    rank, forced = cands[int(rng.integers(0, len(cands)))]
    realised = ids[j]
    rr = next((r + 1 for r, (cid, _) in enumerate(tops[j]) if cid == realised), None)
    info.update({"unforced": False, "j": j, "forced_rank": rank, "forced_id": forced, "forced_text": eng.ttext(forced),
                 "realised_id": realised, "realised_text": texts[j], "realised_rank": rr,
                 "forced_equals_realised": forced == realised, "H_j": H[j],
                 "class_realised": token_class(texts[j], full[:starts[j]]),
                 "class_forced": token_class(eng.ttext(forced), full[:starts[j]]),
                 "cands": [[r, c] for r, c in cands]})
    return info


async def sample(eng: Engine, arm: str, prompt_ids: list[int], *, seed: int, arm_idx: int, call_id: int) -> dict:
    """One call of one arm. Returns completion text, token ids, n tokens, truncation and the DTB/RPB fields."""
    temp, top_p = SAMPLING[arm]
    s1 = call_seed(seed, arm_idx, call_id)
    if arm not in ("DTB", "RPB"):
        g = await eng.complete(prompt_ids, max_tokens=prompts.MAX_TOKENS, temperature=temp, top_p=top_p, seed=s1,
                               logprobs=0)
        return {"ok": g["ok"], "error": g.get("error"), "text": g["text"], "ids": g["ids"], "n_gen_tokens": g["n_tokens"],
                "truncated": g["finish_reason"] == "length", "finish_reason": g["finish_reason"], "vllm_seed": s1,
                "fork": None, "req_tokens": [g["n_tokens"]], "wasted_tokens": 0}
    g1 = await eng.complete(prompt_ids, max_tokens=REQ1_TOKENS, temperature=1.0, top_p=1.0, seed=s1,
                            logprobs=N_LOGPROBS)
    if not g1["ok"]:
        return {"ok": False, "error": g1.get("error"), "text": "", "ids": [], "n_gen_tokens": 0, "truncated": False,
                "finish_reason": None, "vllm_seed": s1, "fork": None, "req_tokens": [0], "wasted_tokens": 0}
    rng = np.random.default_rng([seed, arm_idx, call_id, 7])
    fork = choose_fork(eng, g1, "entropy" if arm == "DTB" else "random", rng)
    s2 = call_seed(seed, arm_idx, call_id, 2)
    if fork.get("unforced"):
        if g1["finish_reason"] == "stop":
            return {"ok": True, "text": g1["text"], "ids": g1["ids"], "n_gen_tokens": g1["n_tokens"], "truncated": False,
                    "finish_reason": "stop", "vllm_seed": s1, "fork": fork, "req_tokens": [g1["n_tokens"]],
                    "wasted_tokens": 0}
        prefix = g1["ids"]
        rest = prompts.MAX_TOKENS - len(prefix)
        g2 = await eng.complete(prompt_ids + prefix, max_tokens=rest, temperature=1.0, top_p=1.0, seed=s2, logprobs=0)
        fork["req2_prompt_len"] = len(prompt_ids) + len(prefix)
    else:
        j = fork["j"]
        prefix = g1["ids"][:j] + [fork["forced_id"]]
        rest = prompts.MAX_TOKENS - j - 1
        g2 = await eng.complete(prompt_ids + prefix, max_tokens=rest, temperature=1.0, top_p=1.0, seed=s2, logprobs=0)
        fork["req2_prompt_len"] = len(prompt_ids) + len(prefix)
    if not g2["ok"]:
        return {"ok": False, "error": g2.get("error"), "text": "", "ids": [], "n_gen_tokens": g1["n_tokens"],
                "truncated": False, "finish_reason": None, "vllm_seed": s1, "fork": fork,
                "req_tokens": [g1["n_tokens"], 0], "wasted_tokens": g1["n_tokens"]}
    ids = prefix + g2["ids"]
    text = eng.tok.decode(prefix) + g2["text"]
    wasted = len(g1["ids"]) - (len(prefix) if fork.get("unforced") else fork["j"])
    return {"ok": True, "text": text, "ids": ids, "n_gen_tokens": g1["n_tokens"] + g2["n_tokens"],
            "truncated": g2["finish_reason"] == "length", "finish_reason": g2["finish_reason"], "vllm_seed": s1,
            "vllm_seed2": s2, "fork": fork, "req_tokens": [g1["n_tokens"], g2["n_tokens"]], "wasted_tokens": wasted}


def entropy_ok(x: float) -> bool:
    return math.isfinite(x)


engine = as_module("engine", "ENGINE_TAG ARMS ARM_IDX SAMPLING REQ1_TOKENS N_LOGPROBS RANK_LO RANK_HI HTTP_TIMEOUT_S "
                             "HTTP_RETRIES call_seed Engine scan_inside body_char_limit token_class entropy_top allowed "
                             "choose_fork sample entropy_ok")

### 5.4 The replay engine: the notebook's only substitute for the model

`ReplayEngine` and `replay_sample` take the place of `engine.Engine` and `engine.sample`. For each call the scheduler
asks for, identified by (seed, arm, call id), they return the completion the original run recorded. That covers its
text, its token ids, and its DTB/RPB fork record. `program_from(text, k)` rebuilds exactly the stored program.

Nothing else is taken from the recording. The scheduler still draws islands and parents from its own database,
renders and tokenises the prompt, evaluates the program in the sandbox, computes F and probe decisions, and
registers the offspring. The per-call vLLM seed is recomputed with `engine.call_seed`. §5.6 compares all of these
with the recorded run. With `USE_LIVE_ENGINE = True`, the original engine talks to a vLLM server instead.

In [ ]:
class ReplayEngine:
    """Drop-in for engine.Engine in replay mode: no HTTP server, no GPU (only what the scheduler touches)."""

    def __init__(self, base_url: str, tokenizer, max_inflight: int = 48, model: str = "qwen3-8b"):
        self.tok, self.model = tokenizer, model
        self.n_http_fail = 0

    async def __aenter__(self):
        return self

    async def __aexit__(self, *exc):
        return None


REPLAY = {(e["metadata_seed"], e["metadata_arm"], e["metadata_call_id"]): e for e in EXAMPLES
          if "replay_round0" in e["metadata_role"]}


async def replay_sample(eng, arm: str, prompt_ids: list[int], *, seed: int, arm_idx: int, call_id: int) -> dict:
    """engine.sample's return value for one call, served from the recorded run."""
    e = REPLAY[(seed, arm, call_id)]
    ok = e["metadata_error_type"] != "http"
    return {"ok": ok, "error": None if ok else e["metadata_error_msg"],
            "text": prompts.body_of(e["output"]) if ok else "",  # program_from(text, k) == the stored program
            "ids": e["metadata_completion_ids"], "n_gen_tokens": e["metadata_n_gen_tokens"],
            "truncated": e["metadata_truncated"], "finish_reason": e["metadata_finish_reason"],
            "vllm_seed": engine.call_seed(seed, arm_idx, call_id),  # recomputed, checked against the recorded seed
            "fork": e["metadata_fork"], "req_tokens": e["metadata_req_tokens"],
            "wasted_tokens": e["metadata_wasted_tokens"]}


if not USE_LIVE_ENGINE:
    engine.Engine, engine.sample = ReplayEngine, replay_sample
print("engine:", "live vLLM at " + VLLM_URL if USE_LIVE_ENGINE else f"replay ({len(REPLAY)} recorded round-0 calls)")

### 5.5 `search.py`: the lock-step scheduler

The scheduler runs every (arm, seed) stream in lock-step rounds of B calls, with a lag-1 pipeline: round r's prompts
are drawn after round r-2's offspring are registered, while round r-1 is still being evaluated. Each round:

1. draw parents and render prompts;
2. generate;
3. submit the programs to the sandbox;
4. finish the previous round: write rows and decision vectors, register valid offspring in their island, and reset
   islands every 400 calls.

Duplicates, identical after AST normalisation, are served from the evaluation cache. The scheduler resumes from
`data/<run>/state.pkl`. The code is verbatim; the module imports are replaced by the namespaces above.

In [ ]:
# ---- search.py
ROOT = bp.ROOT
BF_OR_EXC, BF_W_EXC = 0.05997931747673214, 0.042309606769537086
SEL_SETS = ["SEL-OR", "SEL-W"]
EVAL_TASK = {"sets": SEL_SETS, "limit": 10.0, "fail_fast": True, "probes": ["P1"], "probe_limit": 10.0}
N_PROBE = 2000
ERR = 65535


def utc() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


class Stream:
    def __init__(self, run: str, arm: str, seed: int, target: int, seed_prog: dict, fit: bp.Fitness):
        self.run, self.arm, self.seed, self.target = run, arm, seed, target
        self.arm_idx = engine.ARM_IDX[arm]
        self.name = f"{seed}_{arm}"
        self.n_calls = 0
        self.seen_keys: set[str] = set()
        self.db = None
        if arm != "NS":
            b25 = seed_prog["bins25"]
            self.db = islands.Database(seed, self.arm_idx, seed_prog, islands.SCORE_SCALE * fit.F(b25), tuple(b25))
        self.ff, self.bf = None, None

    def draw(self, ff_prog: dict, bf_prog: dict) -> tuple[int, list[dict]]:
        if self.arm == "NS":
            return -1, [ff_prog, bf_prog]
        return self.db.draw()


class Scheduler:
    def __init__(self, args):
        self.a = args
        self.out = ROOT / "data" / args.run
        self.out.mkdir(parents=True, exist_ok=True)
        (self.out / "decisions").mkdir(exist_ok=True)
        self.fit = bp.Fitness(bp.load_set("SEL-OR"), bp.load_set("SEL-W"), BF_OR_EXC, BF_W_EXC)
        self.state_path = self.out / "state.pkl"
        self.rows_path = self.out / "programs.jsonl"
        self.ids_path = self.out / "completion_ids.jsonl"
        self.status_path = ROOT / "results" / f"partial_status_{args.run}.json"
        self.cache: dict[str, dict] = {}
        self.round_times: list[dict] = []
        bf_b25 = json.loads((ROOT / "results" / "harness_gate.json").read_text())["sel_bf"]
        bins25 = bf_b25["or_bins"] + bf_b25["w_bins"]
        self.bf_prog = {"pid": "seed-BF", "code": bp.SRC_BF, "F": 1.0, "bins25": bins25}
        self.ff_prog = {"pid": "seed-FF", "code": bp.SRC_FF, "F": None, "bins25": None}
        arms, seeds = args.arms.split(","), [int(s) for s in args.seeds.split(",")]
        self.streams = [Stream(args.run, arm, seed, args.rounds, self.bf_prog, self.fit) for seed in seeds for arm in arms]
        self.r = 0
        self.pending = None  # (round, [(stream, call_rec)])

    # ------------------------------------------------------------------ persistence
    def save(self) -> None:
        pend = None if self.pending is None else (self.pending[0], self.pending[1])
        st = {"r": self.r, "streams": self.streams, "pending": pend, "round_times": self.round_times,
              "pause_after": self.a.pause_after}
        tmp = self.state_path.with_suffix(".tmp")
        tmp.write_bytes(pickle.dumps(st))
        tmp.replace(self.state_path)

    def load(self) -> bool:
        if not self.state_path.exists():
            return False
        st = pickle.loads(self.state_path.read_bytes())
        self.r, self.streams, self.round_times = st["r"], st["streams"], st["round_times"]
        self.a.pause_after = st.get("pause_after", self.a.pause_after)
        self.pending = st["pending"]
        if self.rows_path.exists():
            for ln in self.rows_path.read_text().splitlines():
                row = json.loads(ln)
                nk = row.get("norm_key")
                if nk and row["error_type"] not in ("http", "syntax") and nk not in self.cache:
                    dec = self._read_dec(row["dec_ref"]) if row["error_type"] == "ok" else None
                    self.cache[nk] = {"error_type": row["error_type"], "msg": row.get("error_msg", ""),
                                      "F": row.get("F"), "bins25": row.get("bins25"), "eval_s": row.get("eval_s"),
                                      "dec": dec, "probe_failed": row.get("probe_failed", False)}
        logger.info(f"resumed at round {self.r}; {len(self.cache)} cached evaluations; pending "
                    f"{None if self.pending is None else self.pending[0]}")
        return True

    # ------------------------------------------------------------------ control
    def wait_commit2(self) -> None:
        p = ROOT / "predictions" / "commit2.json"
        while not p.exists():
            if self.stop_requested():
                return
            logger.info("paused: waiting for predictions/commit2.json")
            time.sleep(15)
        c2 = json.loads(p.read_text())
        for s in self.streams:
            s.target = int(c2["targets"].get(s.arm, 0))
        logger.info(f"commit #2 targets: {[(s.name, s.target) for s in self.streams]}")
        self.a.pause_after = -1

    def stop_requested(self) -> bool:
        return (ROOT / "STOP").exists() or (ROOT / f"STOP_{self.a.run}").exists() or time.time() >= self.a.deadline

    # ------------------------------------------------------------------ one round
    def draw_round(self, tok, r: int, active: list[Stream]) -> list[tuple]:
        recs = []
        B = self.a.calls_per_round
        for s in active:
            for c in range(B):
                call_id = r * B + c
                isl, parents = s.draw(self.ff_prog, self.bf_prog)
                bodies = [prompts.body_of(p["code"]) for p in parents]
                _, ids = prompts.render(tok, bodies, idea=(s.arm == "IDEA"))
                rec = {"run": self.a.run, "seed": s.seed, "arm": s.arm, "round": r, "call_id": call_id, "island": isl,
                       "k": len(parents), "parent_ids": [p["pid"] for p in parents],
                       "parent_F": [p["F"] for p in parents],
                       "prompt_sha": hashlib.sha256(json.dumps(ids).encode()).hexdigest()[:16],
                       "prompt_len": len(ids), "engine": engine.ENGINE_TAG,
                       "temperature": engine.SAMPLING[s.arm][0], "top_p": engine.SAMPLING[s.arm][1],
                       "pid": f"{self.a.run}-{s.seed}-{s.arm}-{call_id}", "_ids": ids}
                recs.append((s, rec))
        return recs

    async def generate(self, eng: engine.Engine, r: int, recs: list[tuple]) -> None:
        t0 = time.time()
        jobs = [engine.sample(eng, s.arm, rec.pop("_ids"), seed=s.seed, arm_idx=s.arm_idx, call_id=rec["call_id"])
                for s, rec in recs]
        outs = await asyncio.gather(*jobs)
        dt = time.time() - t0
        for (s, rec), o in zip(recs, outs):
            rec["gen"] = o
        logger.info(f"round {r}: generated {len(recs)} calls in {dt:.1f}s "
                    f"({sum(o['n_gen_tokens'] for o in outs) / max(dt, 1e-9):.0f} tok/s)")

    def submit_round(self, pool: sandbox.EvalPool, recs: list[tuple]) -> list:
        futs, inround = [], set()
        for s, rec in recs:
            g = rec["gen"]
            if not g["ok"]:
                rec.update(code=None, norm_key=None)
                futs.append(None)
                continue
            code = prompts.program_from(g["text"], rec["k"])
            nk = sandbox.normalise_key(code)
            rec.update(code=code, norm_key=nk, code_sha256=hashlib.sha256(code.encode()).hexdigest())
            rec["duplicate"] = nk is not None and nk in s.seen_keys
            if nk is not None:
                s.seen_keys.add(nk)
            if nk is None:
                futs.append("syntax")
            elif nk in self.cache or nk in inround:
                rec["cache_hit"] = True
                futs.append(("cached", nk))
            else:
                inround.add(nk)
                futs.append(("new", pool.submit(dict(EVAL_TASK, code=code))))
        return futs

    def finish_round(self, r: int, recs: list[tuple], futs: list) -> None:
        dec_buf: dict[str, list[np.ndarray]] = {}
        rows, idrows = [], []
        for (s, rec), f in zip(recs, futs):
            g = rec.pop("gen")
            dec = np.full(N_PROBE, ERR, dtype=np.uint16)
            if f is None:
                rec.update(error_type="http", error_msg=g.get("error"), F=None, bins25=None, eval_s=0.0)
            elif f == "syntax":
                _, _, msg = sandbox.compile_program(rec["code"])
                rec.update(error_type="syntax", error_msg=msg, F=None, bins25=None, eval_s=0.0)
            else:
                if f[0] == "new":
                    res = f[1].result()
                    c = {"error_type": res.get("error_type", "runtime"), "msg": res.get("msg", ""),
                         "eval_s": res.get("eval_s"), "F": None, "bins25": None, "dec": None, "probe_failed": False}
                    if c["error_type"] == "ok":
                        c["bins25"] = res["bins"]["SEL-OR"] + res["bins"]["SEL-W"]
                        c["F"] = self.fit.F(c["bins25"])
                        pb = res.get("probes", {}).get("P1")
                        if pb is not None:
                            c["dec"] = np.frombuffer(pb, dtype=np.uint16).copy()
                        else:
                            c["probe_failed"] = True
                    self.cache[rec["norm_key"]] = c
                else:
                    c = self.cache[f[1]]
                rec.update(error_type=c["error_type"], error_msg=c["msg"], F=c["F"], bins25=c["bins25"],
                           eval_s=c["eval_s"] if f[0] == "new" else 0.0)
                if c.get("probe_failed"):
                    rec["probe_failed"] = True
                if c["dec"] is not None:
                    dec = c["dec"]
            rec.setdefault("duplicate", False)
            rec.setdefault("cache_hit", False)
            rec["truncated"] = bool(g.get("truncated"))
            rec["n_gen_tokens"] = g.get("n_gen_tokens", 0)
            rec["req_tokens"] = g.get("req_tokens")
            rec["wasted_tokens"] = g.get("wasted_tokens", 0)
            rec["vllm_seed"] = g.get("vllm_seed")
            rec["finish_reason"] = g.get("finish_reason")
            rec["fork"] = g.get("fork")
            fname = f"{s.seed}_{s.arm}.u16"
            rec["dec_ref"] = [fname, s.n_calls]
            dec_buf.setdefault(fname, []).append(dec)
            s.n_calls += 1
            rows.append(rec)
            idrows.append({"pid": rec["pid"], "ids": g.get("ids", [])})
            if s.db is not None and rec["error_type"] == "ok" and rec["F"] is not None:
                prog = {"pid": rec["pid"], "code": rec["code"], "F": rec["F"], "bins25": rec["bins25"]}
                s.db.register(prog, islands.SCORE_SCALE * rec["F"], tuple(rec["bins25"]), rec["island"])
        for fname, decs in dec_buf.items():
            with open(self.out / "decisions" / fname, "ab") as fh:
                np.stack(decs).astype(np.uint16).tofile(fh)
        with open(self.rows_path, "a") as fh:
            for row in rows:
                fh.write(json.dumps(row) + "\n")
        with open(self.ids_path, "a") as fh:
            for row in idrows:
                fh.write(json.dumps(row) + "\n")
        done = {(rec["seed"], rec["arm"]) for _, rec in recs}
        for s in self.streams:
            if s.db is not None and (s.seed, s.arm) in done and s.n_calls % islands.RESET_EVERY == 0:
                ev = s.db.reset()
                logger.info(f"{s.name}: island reset at {s.n_calls} calls: {ev}")

    def _read_dec(self, ref: list) -> np.ndarray:
        fname, i = ref
        a = np.fromfile(self.out / "decisions" / fname, dtype=np.uint16, count=N_PROBE, offset=2 * N_PROBE * int(i))
        return a if a.size == N_PROBE else np.full(N_PROBE, ERR, dtype=np.uint16)

    def status(self, extra: dict | None = None) -> None:
        rows = {}
        for s in self.streams:
            rows[s.name] = {"calls": s.n_calls, "target_rounds": s.target,
                            "best_F": None if s.db is None else -max(s.db.best_score) / 100.0,
                            "clusters": None if s.db is None else s.db.n_clusters()}
        st = {"utc": utc(), "run": self.a.run, "round": self.r, "streams": rows,
              "last_round_times": self.round_times[-3:], **(extra or {})}
        txt = json.dumps(st, indent=1)
        self.status_path.write_text(txt)
        (ROOT / "results" / "partial_status.json").write_text(txt)

    def _finish_pending(self) -> None:
        pr, precs, pfuts = self.pending
        t0 = time.time()
        self.finish_round(pr, precs, pfuts)
        self.pending = None
        logger.info(f"round {pr}: evaluated+registered ({time.time() - t0:.1f}s waiting)")

    # ------------------------------------------------------------------ main loop
    async def run(self) -> None:
        from transformers import AutoTokenizer  # noqa: PLC0415
        tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B-FP8", revision="220b46e3b2180893580a4454f21f22d3ebb187d3")
        resumed = self.load()
        pool = sandbox.EvalPool(self.a.workers, ROOT, as_gib=2.0)
        async with engine.Engine(self.a.url, tok, max_inflight=self.a.inflight) as eng:
            if resumed and self.pending is not None:
                pr, precs = self.pending
                self.pending = (pr, precs, self.submit_round(pool, precs))
            while True:
                active = [s for s in self.streams if s.target > self.r]
                if self.stop_requested() or not active:
                    break
                t0 = time.time()
                # lag-1: round r's prompts are drawn after round r-2's registration (round r-1 still evaluating)
                recs = self.draw_round(tok, self.r, active)
                if self.a.pause_after >= 0 and self.r == self.a.pause_after + 1:
                    if self.pending is not None:
                        self._finish_pending()
                    self.save()
                    self.status({"paused_for_commit2": True})
                    self.wait_commit2()
                    if self.stop_requested():
                        break
                    keep = {s.name for s in self.streams if s.target > self.r}
                    recs = [(s, rec) for s, rec in recs if s.name in keep]
                    if not recs:
                        break
                    t0 = time.time()
                await self.generate(eng, self.r, recs)
                t_gen = time.time() - t0
                futs = self.submit_round(pool, recs)
                t1 = time.time()
                if self.pending is not None:
                    self._finish_pending()
                t_wait = time.time() - t1
                self.pending = (self.r, recs, futs)
                self.round_times.append({"round": self.r, "t_gen": round(t_gen, 2), "t_wait_prev_eval": round(t_wait, 2),
                                         "t_round": round(time.time() - t0, 2), "n_calls": len(recs), "utc": utc(),
                                         "backlog": pool.backlog(), "n_streams": len({s.name for s, _ in recs})})
                self.r += 1
                self.save()
                self.status()
            if self.pending is not None:
                self._finish_pending()
            self.save()
            self.status({"finished": True, "http_failures": eng.n_http_fail})
        pool.close()
        logger.info(f"scheduler finished at round {self.r}: " + ", ".join(f"{s.name}={s.n_calls}" for s in self.streams))

**Run it.** This is the CLI of `search.py main()`, parsed from the config values. The previous run's folder is
cleared first, because the scheduler resumes from `state.pkl`. `asyncio.run(...)` becomes a top-level `await`,
since a notebook already runs an event loop. In replay, generation is instant, so the logged tokens/s is not a
speed; the time goes into sandboxed evaluation. A program that hits the 10 s CPU limit costs the full 10 s.

In [ ]:
# ---- search.py main(): the original arguments
ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True)
ap.add_argument("--arms", default="T1.0,T1.6,DTB,RPB,IDEA")
ap.add_argument("--seeds", default="0,1")
ap.add_argument("--rounds", type=int, default=80)
ap.add_argument("--calls-per-round", type=int, default=25)
ap.add_argument("--pause-after", type=int, default=-1, help="0-based round index after which to wait for commit #2")
ap.add_argument("--deadline", type=float, default=time.time() + 4 * 3600)
ap.add_argument("--workers", type=int, default=10)
ap.add_argument("--inflight", type=int, default=48)
ap.add_argument("--url", default="http://127.0.0.1:8051")
a = ap.parse_args(["--run", "main", "--arms", ",".join(SEARCH_ARMS), "--seeds", ",".join(map(str, SEARCH_SEEDS)),
                   "--rounds", str(ROUNDS), "--calls-per-round", str(CALLS_PER_ROUND), "--url", VLLM_URL])
shutil.rmtree(ROOT / "data" / a.run, ignore_errors=True)  # notebook: start fresh (the scheduler resumes from state.pkl)
logger.info(f"search start {vars(a)}")
t_search = time.time()
await Scheduler(a).run()
print(f"search finished in {time.time() - t_search:.1f}s")

### 5.6 Replay fidelity: recomputed vs recorded

Every row the scheduler just wrote is compared with the original run's row for the same call:

* the island and the parents it drew;
* the prompt's sha256 (Qwen3 chat template with assistant prefill) and its length in tokens;
* the recomputed vLLM seed and the program's sha256;
* the error type, the 25 selection bin counts and F;
* the sha256 of the 2,000-state decision vector.

A mismatch would mean the pipeline here differs from the one that ran. A program near the 10 s CPU limit can be the
exception: a faster or slower CPU can turn a timeout into a pass, or the reverse.

In [ ]:
replayed = [json.loads(ln) for ln in (ROOT / "data" / "main" / "programs.jsonl").read_text().splitlines()]
recorded = {e["metadata_pid"]: e for e in EXAMPLES}
checks, mismatch = Counter(), []
for r in replayed:
    e = recorded[r["pid"]]
    inp = json.loads(e["input"])
    with open(ROOT / "data" / "main" / "decisions" / r["dec_ref"][0], "rb") as fh:
        fh.seek(2 * N_PROBE * r["dec_ref"][1])
        dec_sha = hashlib.sha256(fh.read(2 * N_PROBE)).hexdigest()
    rec_F = float(e["predict_F"]) if e["predict_F"] else None
    same = {"island + parents drawn": r["island"] == inp["island"] and r["parent_ids"] == inp["parent_ids"],
            "prompt sha256 + length": r["prompt_sha"] == inp["prompt_sha"] and r["prompt_len"] == e["metadata_prompt_len"],
            "vLLM per-call seed": r["vllm_seed"] == e["metadata_vllm_seed"],
            "program sha256": r["code_sha256"] == e["metadata_code_sha256"],
            "error type": r["error_type"] == e["metadata_error_type"],
            "25 bin counts + F": r["bins25"] == e["metadata_bins25"] and r["F"] == rec_F,
            "P1 decision vector sha256": dec_sha == e["metadata_dec_sha256"]}
    checks.update(k for k, v in same.items() if v)
    if not all(same.values()):
        mismatch.append((r["pid"], [k for k, v in same.items() if not v], r["error_type"], e["metadata_error_type"]))
print(f"{len(replayed)} replayed calls ({', '.join(f'{k}: {v}' for k, v in Counter(r['arm'] for r in replayed).items())})")
for k in same:
    print(f"  {k:28s} {checks[k]:3d}/{len(replayed)} identical to the recorded run")
for m in mismatch:
    print("  differs:", m)
print("error types:", dict(Counter(r["error_type"] for r in replayed)),
      "| cache hits:", sum(r.get("cache_hit", False) for r in replayed),
      "| best F this round:", min((r["F"] for r in replayed if r["F"] is not None), default=None))

### 5.7 The proposed operator: re-deriving every DTB / RPB fork

Each DTB and RPB row records REQ1's per-token entropies `H`, the eligible positions `E`, and the candidates at
ranks 2-8 at the chosen position. The cell below re-applies `choose_fork`'s selection step to those records, using
the call's own generator `default_rng([seed, arm_idx, call_id, 7])`, and recovers both the fork position and the
forced token. The cell also decodes each completion with the Qwen3 tokenizer, to show where the fork landed.

In [ ]:
from transformers import AutoTokenizer  # the scheduler loads it inside run(); loaded again here to decode
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B-FP8", revision="220b46e3b2180893580a4454f21f22d3ebb187d3")
ids_by_pid = {json.loads(ln)["pid"]: json.loads(ln)["ids"]
              for ln in (ROOT / "data" / "main" / "completion_ids.jsonl").read_text().splitlines()}
fork_rows = [r for r in replayed if r["arm"] in ("DTB", "RPB") and r.get("fork") and not r["fork"].get("unforced")]
n_j = n_tok = 0
for r in fork_rows:
    f = r["fork"]
    rng = np.random.default_rng([r["seed"], engine.ARM_IDX[r["arm"]], r["call_id"], 7])
    if r["arm"] == "DTB":
        j_idx = int(np.argmax([f["H"][t] for t in f["E"]]))  # choose_fork: highest top-20 entropy, earliest on ties
    else:
        j_idx = int(rng.integers(0, len(f["E"])))  # choose_fork: uniform over the eligible positions
    j = f["E"][j_idx]
    rank, forced = f["cands"][int(rng.integers(0, len(f["cands"])))]  # uniform over the allowed ranks 2..8 at j
    n_j += j == f["j"]
    n_tok += j == f["j"] and rank == f["forced_rank"] and forced == f["forced_id"]
print(f"{len(fork_rows)} forced calls: fork position re-derived {n_j}/{len(fork_rows)}, "
      f"forced token re-derived {n_tok}/{len(fork_rows)}")
for r in [x for x in fork_rows if x["arm"] == "DTB"][:3] + [x for x in fork_rows if x["arm"] == "RPB"][:2]:
    f = r["fork"]
    prefix = tok.decode(ids_by_pid[r["pid"]][:f["j"]])
    print(f"\n{r['arm']} call {r['call_id']}: fork at token {f['j']} of {len(f['H'])} (H_j = {f['H_j']:.2f}, "
          f"max H over E = {max(f['H'][t] for t in f['E']):.2f})")
    print(f"   ...{prefix!r} + [rank-1 {f['realised_text']!r} -> forced rank-{f['forced_rank']} {f['forced_text']!r}]"
          f"  ({f['class_realised']} -> {f['class_forced']})")
    print("   program:", " ".join(prompts.body_of(r["code"]).split())[:110], f"| F = {r['F']}")

## 6. Stage `post`: behaviour clusters and the verdict rules

### 6.1 `cluster.py`: behaviour clusters

`agreement(a, b)` is the share of the 2,000 probe states on which two programs pick the same bin. Complete linkage
on the unique decision vectors is cut at 100 disagreeing states, so every pair inside a cluster agrees on at least
95 % of states. Leader clustering in call order is the check. AST-shingle *surface* clusters count different-looking
code.

In [ ]:
# ---- cluster.py
ERR = 65535
N_STATES = 2000
THRESH = 0.95
CUT = int(round(N_STATES * (1 - THRESH)))  # 100 disagreeing states


def agree_counts(U: np.ndarray, V: np.ndarray | None = None) -> np.ndarray:
    """C[i, j] = number of states where U_i == V_j and U_i != ERR (int32)."""
    V = U if V is None else V
    U = np.ascontiguousarray(U, dtype=np.uint16)
    V = np.ascontiguousarray(V, dtype=np.uint16)
    out = np.zeros((len(U), len(V)), dtype=np.int32)
    if len(U) == 0 or len(V) == 0:
        return out
    blk = max(1, int(1.5e8 // max(1, len(V) * U.shape[1])))
    for i0 in range(0, len(U), blk):
        u = U[i0:i0 + blk]
        eq = (u[:, None, :] == V[None, :, :]) & (u[:, None, :] != ERR)
        out[i0:i0 + blk] = eq.sum(axis=2, dtype=np.int32)
    return out


def agreement(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a), np.asarray(b)
    return float(np.mean((a == b) & (a != ERR)))


def unique_counts(D: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Unique decision vectors, the inverse map row -> unique index, and their pairwise agreement counts."""
    if len(D) == 0:
        return np.zeros((0, N_STATES), dtype=np.uint16), np.zeros(0, dtype=int), np.zeros((0, 0), dtype=np.int32)
    uniq, inv = np.unique(D, axis=0, return_inverse=True)
    return uniq, inv.reshape(-1), agree_counts(uniq)


def linkage_labels(C: np.ndarray) -> tuple[np.ndarray, int]:
    """Complete linkage on unique vectors given their agreement counts; cut so all pairs agree >= 0.95."""
    n = len(C)
    if n == 0:
        return np.zeros(0, dtype=int), 0
    if n == 1:
        return np.zeros(1, dtype=int), 1
    dist = (N_STATES - C).astype(np.float64)
    np.fill_diagonal(dist, 0.0)
    dist = np.maximum(dist, dist.T)
    iu = np.triu_indices(n, 1)
    lab = fcluster(linkage(dist[iu], method="complete"), t=CUT, criterion="distance")
    _, lab = np.unique(lab, return_inverse=True)
    return lab.reshape(-1), int(lab.max() + 1)


def clusters_in_subset(inv: np.ndarray, C: np.ndarray, rows: np.ndarray) -> int:
    """Complete-linkage cluster count recomputed on a subset of rows (prefixes, rarefied subsets, shuffles)."""
    if len(rows) == 0:
        return 0
    u = np.unique(inv[rows])
    return linkage_labels(C[np.ix_(u, u)])[1]


def complete_linkage(D: np.ndarray, pre: tuple | None = None) -> tuple[np.ndarray, int]:
    """Cluster labels (per row of D) with complete linkage cut so that all within-cluster pairs agree >= 0.95.
    Unique vectors are clustered once (exact duplicates map to one point, as the plan's 'unique vectors with
    multiplicities')."""
    if len(D) == 0:
        return np.zeros(0, dtype=int), 0
    uniq, inv, C = pre if pre is not None else unique_counts(D)
    lab_u, n = linkage_labels(C)
    lab = lab_u[inv]
    _, lab = np.unique(lab, return_inverse=True)
    return lab.reshape(-1), int(lab.max() + 1)


def leader(D: np.ndarray) -> tuple[np.ndarray, list[int]]:
    """Leader clustering in row (call) order; a row joins the first leader it agrees with >= 0.95."""
    n = len(D)
    L = np.empty((n, D.shape[1] if D.ndim == 2 else N_STATES), dtype=np.uint16)
    leaders: list[int] = []
    lab = np.zeros(n, dtype=int)
    for i in range(n):
        k = len(leaders)
        if k:
            c = ((L[:k] == D[i]) & (L[:k] != ERR)).sum(axis=1)
            hit = np.nonzero(c >= N_STATES - CUT)[0]
            if hit.size:
                lab[i] = int(hit[0])
                continue
        L[k] = D[i]
        leaders.append(i)
        lab[i] = k
    return lab, leaders


def discovery_curves(D: np.ndarray, call_idx: np.ndarray, n_calls: int, step: int = 100,
                     pre: tuple | None = None) -> dict:
    """Complete-linkage cluster counts on prefixes every `step` calls and the leader curve (per call)."""
    uniq, inv, C = pre if pre is not None else unique_counts(D)
    pts, cl = [], []
    for c in range(step, n_calls + step, step):
        rows = np.nonzero(call_idx < c)[0]
        pts.append(min(c, n_calls))
        cl.append(clusters_in_subset(inv, C, rows))
    lab, leaders = leader(D)
    first = np.zeros(n_calls, dtype=int)
    for li in leaders:
        first[int(call_idx[li])] += 1
    return {"calls": pts, "complete": cl, "leader_cum": np.cumsum(first).tolist()}


# ------------------------------------------------------------------------------------------- surface (AST) clusters
def shingles(code: str, n: int = 4) -> set[str] | None:
    try:
        tree = ast.parse(code)
    except (SyntaxError, ValueError):
        return None
    toks = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Name):
            toks.append("ID")
        elif isinstance(node, ast.Constant):
            toks.append("CONST")
        elif isinstance(node, ast.Attribute):
            toks.append("ATTR:" + (node.attr if re.fullmatch(r"[a-z_]+", node.attr) else "x"))
        elif isinstance(node, ast.arg):
            toks.append("ARG")
        else:
            toks.append(type(node).__name__)
    return {"|".join(toks[i:i + n]) for i in range(max(1, len(toks) - n + 1))}


def surface_clusters(codes: list[str], thresh: float = 0.8) -> tuple[np.ndarray, int]:
    sets = [shingles(c) or set() for c in codes]
    vocab: dict[str, int] = {}
    rows, cols = [], []
    for i, s in enumerate(sets):
        for t in s:
            cols.append(vocab.setdefault(t, len(vocab)))
            rows.append(i)
    if not codes:
        return np.zeros(0, dtype=int), 0
    A = csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(codes), max(1, len(vocab))))
    inter = (A @ A.T).toarray()
    sz = np.array([len(s) for s in sets], dtype=np.float32)
    union = sz[:, None] + sz[None, :] - inter
    J = np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0)
    adj = csr_matrix(J >= thresh)
    n, lab = connected_components(adj, directed=False)
    return lab, int(n)


cluster = as_module("cluster", "ERR N_STATES THRESH CUT agree_counts agreement unique_counts linkage_labels "
                               "clusters_in_subset complete_linkage leader discovery_curves shingles surface_clusters")

### 6.2 `analysis.py`: the committed verdict rules

These rules were hash-committed before the samples they govern:

* **V1** passes an arm when its best program is at most 0.8 x best fit's excess on both H-FSW5k and H-OR3.
* **V2** compares cluster rates per 1,000 calls: R1 = DTB/T1.0 >= 1.5, R2 = DTB/T1.6 >= 1.5 and R3 = DTB/RPB >= 1.2.
* **V3** makes a program a novelty candidate when it passes both held-out bars and agrees with every known heuristic
  on fewer than 90 % of states.
* **V6** measures how often IDEA renames its parents (agreement >= 0.95 with a parent).

The module is verbatim (`import cluster` is replaced by the namespace above).

In [ ]:
# ---- analysis.py (ROOT from the config cell)
BF_FSW5K, BF_OR3 = 3.9843042559613666, 5.367793240556666
THR_FSW5K, THR_OR3 = 0.8 * BF_FSW5K, 0.8 * BF_OR3
CORE = ["T1.0", "T1.6", "DTB", "RPB"]
THRESHOLDS = {"v1_rule": 0.8, "R1_R2": 1.5, "R3": 1.2, "per_seed": 1.0, "disconfirm": 1.0, "cluster": 0.95,
              "nb_corpus": 0.9, "v6_support": 0.4, "v6_disconfirm": 0.2, "v6_margin": 0.1}
NEW_IDEA_RX = r"new|novel|different|alternative|instead|unlike|another approach|fresh"


# ----------------------------------------------------------------------------------------------- loaders
def load_rows(run: str) -> list[dict]:
    p = ROOT / "data" / run / "programs.jsonl"
    return [json.loads(ln) for ln in p.read_text().splitlines()] if p.exists() else []


def load_decisions(run: str, seed: int, arm: str) -> np.ndarray:
    p = ROOT / "data" / run / "decisions" / f"{seed}_{arm}.u16"
    a = np.fromfile(p, dtype=np.uint16) if p.exists() else np.zeros(0, dtype=np.uint16)
    return a.reshape(-1, cluster.N_STATES)


def stream_rows(rows: list[dict], seed: int, arm: str, c_s: int | None = None) -> list[dict]:
    rs = sorted((r for r in rows if r["seed"] == seed and r["arm"] == arm), key=lambda r: r["call_id"])
    return rs if c_s is None else [r for r in rs if r["call_id"] < c_s]


def valid(r: dict) -> bool:
    return r["error_type"] == "ok" and r.get("F") is not None


# ----------------------------------------------------------------------------------------------- V1
def v1(best: dict[str, dict]) -> dict:
    """best[arm] = {'exc_FSW5k': %, 'exc_OR3': %, ...} for the better-by-F of the arm's two seed-bests."""
    per = {}
    for a in CORE:
        b = best.get(a) or {}
        fs, o3 = b.get("exc_FSW5k"), b.get("exc_OR3")
        p_fs = fs is not None and fs <= THR_FSW5K
        p_o3 = o3 is not None and o3 <= THR_OR3
        per[a] = {"exc_FSW5k": fs, "exc_OR3": o3, "pass_FSW5k": p_fs, "pass_OR3": p_o3, "pass": p_fs and p_o3}
    if all(v["pass"] for v in per.values()):
        verdict = "SUPPORTED"
    elif not any(v["pass_FSW5k"] or v["pass_OR3"] for v in per.values()):
        verdict = "DISCONFIRMED"
    else:
        verdict = "PARTIAL"
    return {"verdict": verdict, "per_arm": per, "thr_FSW5k_pct": THR_FSW5K, "thr_OR3_pct": THR_OR3}


# ----------------------------------------------------------------------------------------------- V2
def rho(b: dict) -> float | None:
    if b.get("exc_FSW5k") is None or b.get("exc_OR3") is None:
        return None
    return 0.5 * (b["exc_FSW5k"] / BF_FSW5K + b["exc_OR3"] / BF_OR3)


def v2(D: dict[tuple[str, int], int], C: dict[int, int], best: dict[str, dict]) -> dict:
    """D[(arm, seed)] = complete-linkage clusters in stream (arm, seed); C[seed] = completed calls per stream."""
    rate = {a: 1000.0 * (D[(a, 0)] + D[(a, 1)]) / (C[0] + C[1]) for a in CORE}
    R1 = rate["DTB"] / rate["T1.0"] if rate["T1.0"] else math.inf
    R2 = rate["DTB"] / rate["T1.6"] if rate["T1.6"] else math.inf
    R3 = rate["DTB"] / rate["RPB"] if rate["RPB"] else math.inf
    r = {f"r_s{s}_{X}": (D[("DTB", s)] / D[(X, s)] if D[(X, s)] else math.inf) for s in (0, 1) for X in ("T1.0", "T1.6")}
    rh = {a: rho(best.get(a) or {}) for a in CORE}
    conds = {"R1>=1.5": R1 >= 1.5, "R2>=1.5": R2 >= 1.5, "all_per_seed_r>1": all(v > 1 for v in r.values()),
             "R3>=1.2": R3 >= 1.2,
             "rho_DTB<=min(rho_T1.0,rho_T1.6)": (rh["DTB"] is not None and rh["T1.0"] is not None and
                                                  rh["T1.6"] is not None and rh["DTB"] <= min(rh["T1.0"], rh["T1.6"]))}
    if all(conds.values()):
        verdict = "SUPPORTED"
    elif R1 <= 1.0 or R2 <= 1.0:
        verdict = "DISCONFIRMED"
    else:
        verdict = "PARTIAL"
    return {"verdict": verdict, "rates_per_1000_calls": rate, "R1": R1, "R2": R2, "R3": R3, "per_seed": r,
            "rho": rh, "conditions": conds, "failed": [k for k, v in conds.items() if not v]}


def round_block_bootstrap(labels: dict[tuple[str, int], np.ndarray], rounds: dict[tuple[str, int], np.ndarray],
                          n_rounds: dict[int, int], B: int = 2000, seed: int = 20261026) -> dict:
    """Paired round-block bootstrap: the same resampled round indices for all arms within a seed; the count is the
    number of distinct full-run cluster labels among the resampled rows."""
    rng = np.random.default_rng(seed)
    out = {k: np.zeros(B) for k in ("R1", "R2", "R3")}
    for b in range(B):
        res = {s: rng.integers(0, n_rounds[s], size=n_rounds[s]) for s in (0, 1)}
        cnt = {}
        for a in CORE:
            tot = 0
            for s in (0, 1):
                lab, rd = labels[(a, s)], rounds[(a, s)]
                picked = np.isin(rd, res[s])
                tot += len(np.unique(lab[picked])) if picked.any() else 0
            cnt[a] = tot
        out["R1"][b] = cnt["DTB"] / cnt["T1.0"] if cnt["T1.0"] else np.nan
        out["R2"][b] = cnt["DTB"] / cnt["T1.6"] if cnt["T1.6"] else np.nan
        out["R3"][b] = cnt["DTB"] / cnt["RPB"] if cnt["RPB"] else np.nan
    return {k: {"p2.5": float(np.nanpercentile(v, 2.5)), "p50": float(np.nanpercentile(v, 50)),
                "p97.5": float(np.nanpercentile(v, 97.5))} for k, v in out.items()}


def rarefied(decs: dict[str, np.ndarray], n: int, reps: int = 200, seed: int = 20261027) -> dict[str, float]:
    """Mean complete-linkage cluster count over `reps` random subsets of size n (valid offspring) per arm."""
    rng = np.random.default_rng(seed)
    out = {}
    for a, D in decs.items():
        if len(D) < n or n == 0:
            out[a] = float("nan")
            continue
        out[a] = float(np.mean([cluster.complete_linkage(D[rng.choice(len(D), n, replace=False)])[1]
                                for _ in range(reps)]))
    return out


def shuffle_placebo(lab_a: dict[int, np.ndarray], lab_b: dict[int, np.ndarray], decs_a: dict[int, np.ndarray],
                    decs_b: dict[int, np.ndarray], n_shuffles: int = 200, seed: int = 20261028) -> dict:
    """Within-seed arm-label shuffles of DTB and T1.0 rows: how often does the shuffled R1 reach 1.5?"""
    rng = np.random.default_rng(seed)
    hits, vals = 0, []
    for _ in range(n_shuffles):
        da, db = 0, 0
        for s in (0, 1):
            pool = np.concatenate([decs_a[s], decs_b[s]])
            perm = rng.permutation(len(pool))
            na = len(decs_a[s])
            da += cluster.complete_linkage(pool[perm[:na]])[1]
            db += cluster.complete_linkage(pool[perm[na:]])[1]
        r1 = da / db if db else np.nan
        vals.append(r1)
        hits += int(r1 >= 1.5)
    return {"n": n_shuffles, "share_R1_ge_1.5": hits / n_shuffles, "R1_shuffled_p50": float(np.nanmedian(vals)),
            "R1_shuffled_p97.5": float(np.nanpercentile(vals, 97.5))}


# ----------------------------------------------------------------------------------------------- V3
def v3(audited: list[dict], n_nb: int) -> dict:
    """audited: representatives with final 'cls' in {NOVEL, KNOWN, KNOWN_FAMILY_NEW_PARAMETERS, DEGENERATE}."""
    novel = [c for c in audited if c.get("cls") == "NOVEL"]
    n = len(audited)
    out = {"n_nb_set": n_nb, "n_audited": n, "n_novel": len(novel), "novel_ids": [c["anon_id"] for c in novel]}
    if novel:
        out["verdict"] = "SUPPORTED"
    else:
        out["verdict"] = "DISCONFIRMED"
        out["rule_of_three_upper_95"] = (3.0 / n) if n else None
    return out


# ----------------------------------------------------------------------------------------------- V6
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


def newcombe(k1: int, n1: int, k2: int, n2: int) -> tuple[float, float]:
    """Newcombe (1998) hybrid-score CI for p1 - p2."""
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wilson(k1, n1)
    l2, u2 = wilson(k2, n2)
    d = p1 - p2
    return (d - math.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), d + math.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2))


def v6(k_idea: int, n_idea: int, k_t10: int, n_t10: int) -> dict:
    r_i = k_idea / n_idea if n_idea else float("nan")
    r_t = k_t10 / n_t10 if n_t10 else float("nan")
    if r_i >= 0.4 and r_i >= r_t - 0.1:
        verdict = "SUPPORTED"
    elif r_i < 0.2:
        verdict = "DISCONFIRMED"
    else:
        verdict = "PARTIAL"
    return {"verdict": verdict, "rate_IDEA": r_i, "rate_T1.0": r_t, "n_IDEA": n_idea, "n_T1.0": n_t10,
            "wilson_IDEA": wilson(k_idea, n_idea), "wilson_T1.0": wilson(k_t10, n_t10),
            "newcombe_IDEA_minus_T1.0": newcombe(k_idea, n_idea, k_t10, n_t10) if n_idea and n_t10 else None}


analysis = as_module("analysis", "ROOT BF_FSW5K BF_OR3 THR_FSW5K THR_OR3 CORE THRESHOLDS NEW_IDEA_RX load_rows "
                                 "load_decisions stream_rows valid v1 rho v2 round_block_bootstrap rarefied "
                                 "shuffle_placebo v3 wilson newcombe v6")

### 6.3 `postsearch.py stage_clusters`: clusters per stream

For each replayed stream, the cell counts:

* valid offspring;
* complete-linkage behaviour clusters (`D`) and leaders;
* surface (code) clusters;
* clusters among offspring that beat best fit.

The functions are verbatim. `SEEDS` and `ARMS_MAIN` come from the config, and `c_s_main` takes the minimum over the
replayed arms.

In [ ]:
# ---- postsearch.py (helpers and stages used below; ROOT from the config cell)
RES = ROOT / "results"
SEEDS = tuple(SEARCH_SEEDS)     # notebook: from the config (original (0, 1))
ARMS_MAIN = list(SEARCH_ARMS)   # notebook: from the config (original ["T1.0", "T1.6", "DTB", "RPB", "IDEA"])
HELDOUT_ALL = ["H-FSW5k", "H-OR3", "H-OR1", "H-OR2", "H-OR4", "H-W5k", "H-W10k", "H-SHIFT"]
N_WORKERS = 10


def utc() -> str:
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def wjson(name: str, obj) -> None:
    (RES / name).write_text(json.dumps(obj, indent=1, default=float))


def rjson(name: str):
    return json.loads((RES / name).read_text())


def heldout_pct(set_name: str, bins: list[int]) -> float:
    return 100.0 * bp.family_excess(bins, [x["l1"] for x in bp.load_set(set_name)])


def streams() -> list[tuple[str, str, int]]:
    out = [("main", a, s) for s in SEEDS for a in ARMS_MAIN]
    if (ROOT / "data" / "ns" / "programs.jsonl").exists():
        out += [("ns", "NS", s) for s in SEEDS]
    return out


def load_stream(run: str, arm: str, seed: int, c_s: int | None) -> tuple[list[dict], np.ndarray]:
    rows = analysis.stream_rows(analysis.load_rows(run), seed, arm, c_s)
    D = analysis.load_decisions(run, seed, arm)[:len(rows)]
    return rows, D


def c_s_main() -> int:
    rows = analysis.load_rows("main")
    n = {}
    for r in rows:
        n[(r["arm"], r["seed"])] = n.get((r["arm"], r["seed"]), 0) + 1
    core = [n.get((a, s), 0) for a in ARMS_MAIN for s in SEEDS]  # notebook: replayed arms (original analysis.CORE)
    return min(core)


# ----------------------------------------------------------------------------------------------- (a, b) clusters
def stage_clusters() -> dict:
    t0 = time.time()
    c_s = c_s_main()
    ns_rows = analysis.load_rows("ns")
    c_ns = min([sum(1 for r in ns_rows if r["seed"] == s) for s in SEEDS]) if ns_rows else 0
    out = {"utc": utc(), "C_s": c_s, "C_ns": c_ns, "streams": {}}
    for run, arm, seed in streams():
        cs = c_ns if run == "ns" else c_s
        if arm == "IDEA":
            cs = min(cs, sum(1 for r in analysis.load_rows("main") if r["arm"] == "IDEA" and r["seed"] == seed))
        rows, D = load_stream(run, arm, seed, cs)
        vm = np.array([analysis.valid(r) and not r.get("probe_failed") for r in rows], dtype=bool)
        n_probe_failed = int(sum(analysis.valid(r) and r.get("probe_failed", False) for r in rows))
        Dv = D[vm]
        pre = cluster.unique_counts(Dv)
        lab, n = cluster.complete_linkage(Dv, pre)
        llab, leaders = cluster.leader(Dv)
        call_idx = np.array([r["call_id"] for r, m in zip(rows, vm) if m])
        curves = cluster.discovery_curves(Dv, call_idx, cs, step=100, pre=pre)
        codes = [r["code"] for r, m in zip(rows, vm) if m]
        slab, ns_ = cluster.surface_clusters(codes)
        F = np.array([r["F"] for r, m in zip(rows, vm) if m])
        lab_f1 = lab[F < 1]
        toks = sum(r["n_gen_tokens"] for r in rows)
        out["streams"][f"{run}:{seed}_{arm}"] = {
            "run": run, "arm": arm, "seed": seed, "calls": len(rows), "n_valid": int(vm.sum()),
            "n_probe_failed_singletons": n_probe_failed, "D": int(n + n_probe_failed),
            "D_complete_only": int(n), "leaders": len(leaders), "surface_clusters": int(ns_),
            "surface_per_behaviour": ns_ / max(1, n), "D_F_lt_1": int(len(np.unique(lab_f1))),
            "n_F_lt_1": int((F < 1).sum()), "gen_tokens": int(toks),
            "clusters_per_million_tokens": 1e6 * (n + n_probe_failed) / max(1, toks),
            "validity": float(np.mean([analysis.valid(r) for r in rows])) if rows else None,
            "curves": curves, "labels": lab.tolist(), "valid_call_ids": call_idx.tolist(),
            "leader_labels": llab.tolist(), "best_F": float(F.min()) if len(F) else None}
        logger.info(f"{run}:{seed}_{arm}: calls {len(rows)} valid {int(vm.sum())} D {n} leaders {len(leaders)} "
                    f"surface {ns_} ({time.time() - t0:.0f}s)")
    wjson("clusters.json", out)
    wjson("curves.json", {k: {"curves": v["curves"], "arm": v["arm"], "seed": v["seed"]} for k, v in out["streams"].items()})
    return out


# ----------------------------------------------------------------------------------------------- (c) best programs
def eval_heldout(codes: list[str], sets: list[str], limit: float = 120.0) -> list[dict]:
    pool = sandbox.EvalPool(N_WORKERS, ROOT, as_gib=2.0)
    futs = [pool.submit({"code": c, "sets": sets, "limit": limit, "fail_fast": False, "probes": []}) for c in codes]
    res = [f.result() for f in futs]
    pool.close()
    out = []
    for r in res:
        out.append({"heldout_excess_pct": {s: heldout_pct(s, b) for s, b in r["bins"].items()},
                    "heldout_bins": r["bins"], "error": None if r["error_type"] == "ok" else
                    f"{r['error_type']}: {r.get('msg', '')}"})
    return out


# ----------------------------------------------------------------------------------------------- V1 / V2
def best_of_arm(best: dict, arm: str) -> dict | None:
    cands = [v for v in best.values() if v["arm"] == arm]
    if not cands:
        return None
    b = min(cands, key=lambda v: (v["F"], len(v["code"])))
    ho = b.get("heldout_excess_pct", {})
    return {"pid": b["pid"], "F": b["F"], "exc_FSW5k": ho.get("H-FSW5k"), "exc_OR3": ho.get("H-OR3"),
            "heldout_excess_pct": ho, "seed": b["seed"]}


# ----------------------------------------------------------------------------------------------- V3
def corpus_vectors() -> tuple[list[str], np.ndarray, list[dict]]:
    c = json.loads((ROOT / "data" / "corpus" / "corpus.json").read_text())
    ents = [e for e in c["entries"] if e.get("decisions") is not None]
    return [e["name"] for e in ents], np.array([e["decisions"] for e in ents], dtype=np.uint16), ents


# ----------------------------------------------------------------------------------------------- V6 / evaluator
def comment_text(code: str) -> str:
    parts = re.findall(r"#[^\n]*", code)
    parts += re.findall(r'"""(.*?)"""', code, re.S) + re.findall(r"'''(.*?)'''", code, re.S)
    return " ".join(parts)


def stage_v6() -> dict:
    names, CV, _ = corpus_vectors()
    bf_dec = CV[names.index("BF")]
    ff_dec = CV[names.index("FF")]
    rows = analysis.load_rows("main")
    decs = {(a, s): analysis.load_decisions("main", s, a) for a in ARMS_MAIN for s in SEEDS}
    pid_dec = {}
    for r in rows:
        f, i = r["dec_ref"]
        seed, arm = f[:-4].split("_", 1)
        pid_dec[r["pid"]] = (arm, int(seed), i)

    def dec_of(pid: str) -> np.ndarray:
        if pid == "seed-BF":
            return bf_dec
        if pid == "seed-FF":
            return ff_dec
        a, s, i = pid_dec[pid]
        return decs[(a, s)][i]

    cl = rjson("clusters.json")
    out = {}
    rx = re.compile(analysis.NEW_IDEA_RX, re.I)
    for arm in ARMS_MAIN:
        k = n = claims = 0
        for s in SEEDS:
            cs = cl["streams"][f"main:{s}_{arm}"]["calls"]
            for r in analysis.stream_rows(rows, s, arm, cs):
                if not analysis.valid(r) or r.get("probe_failed"):
                    continue
                d = dec_of(r["pid"])
                ag = max(cluster.agreement(d, dec_of(p)) for p in r["parent_ids"])
                k += int(ag >= 0.95)
                n += 1
                claims += int(bool(rx.search(comment_text(r["code"]))))
        out[arm] = {"renames": k, "n_valid": n, "rate": k / max(1, n), "new_idea_claim_share": claims / max(1, n),
                    "wilson": analysis.wilson(k, n)}
    if "IDEA" not in out or "T1.0" not in out:  # notebook: the verdict needs both arms in the replay
        return {"per_arm": out}
    v6 = analysis.v6(out["IDEA"]["renames"], out["IDEA"]["n_valid"], out["T1.0"]["renames"], out["T1.0"]["n_valid"])
    v6["per_arm"] = out
    # NS (fixed FF/BF parents) for reference
    ns_rows = analysis.load_rows("ns")
    if ns_rows:
        k = n = 0
        nsd = {s: analysis.load_decisions("ns", s, "NS") for s in SEEDS}
        for r in ns_rows:
            if not analysis.valid(r) or r.get("probe_failed"):
                continue
            d = nsd[r["seed"]][r["dec_ref"][1]]
            k += int(max(cluster.agreement(d, bf_dec), cluster.agreement(d, ff_dec)) >= 0.95)
            n += 1
        v6["NS_reference"] = {"renames": k, "n_valid": n, "rate": k / max(1, n)}
    wjson("v6.json", v6)
    logger.info(f"V6 {v6['verdict']}: IDEA {v6['rate_IDEA']:.3f} T1.0 {v6['rate_T1.0']:.3f}")
    return v6


cl = stage_clusters()
print(f"\n{'stream':14s} {'calls':>5s} {'valid':>5s} {'D':>3s} {'leaders':>7s} {'surface':>7s} {'best F':>8s}")
for key, st in cl["streams"].items():
    print(f"{key:14s} {st['calls']:5d} {st['n_valid']:5d} {st['D']:3d} {st['leaders']:7d} {st['surface_clusters']:7d} "
          f"{st['best_F'] if st['best_F'] is None else round(st['best_F'], 4):>8}")

### 6.4 V2 at demo scale

V2 compares behaviour clusters per 1,000 calls across arms. Round 0 is the hardest place to see a difference: every
prompt is identical, with best fit as the only parent, and most offspring re-implement best fit. The cell computes
the rates and ratios with `analysis.v2`'s formulas over the replayed streams, next to the full run's numbers.
The verdict itself, `analysis.v2`, needs both seeds and 2,000 calls per arm.

In [ ]:
st = cl["streams"]
calls_demo = {a: sum(st[f"main:{s}_{a}"]["calls"] for s in SEEDS) for a in ARMS_MAIN}
rate_demo = {a: 1000.0 * sum(st[f"main:{s}_{a}"]["D"] for s in SEEDS) / calls_demo[a] for a in ARMS_MAIN}
ref2 = data["metadata"]["reference"]["v2"]
full_D = {a: ref2["D"][f"{a}|0"] + ref2["D"][f"{a}|1"] for a in analysis.CORE}
full_D["IDEA"] = sum(data["metadata"]["reference"]["v2_IDEA_clusters"].values())
print(f"{'arm':5s} | {'demo: calls':>11s} {'clusters':>8s} {'per 1k calls':>12s} | {'full run: clusters':>18s} "
      f"{'per 1k calls':>12s}  (2 x {ref2['C_s']} calls)")
for a in ARMS_MAIN:
    print(f"{a:5s} | {calls_demo[a]:11d} {sum(st[f'main:{s}_{a}']['D'] for s in SEEDS):8d} {rate_demo[a]:12.1f} | "
          f"{full_D[a]:18d} {1000.0 * full_D[a] / (2 * ref2['C_s']):12.1f}")
for name, d in (("R1", "T1.0"), ("R2", "T1.6"), ("R3", "RPB")):
    demo = f"{rate_demo['DTB'] / rate_demo[d]:.2f}" if "DTB" in rate_demo and d in rate_demo else "n/a"
    ci = ref2["bootstrap"][name]
    print(f"{name} = DTB/{d:4s}: demo {demo:>5s} | full run {ref2[name]:.2f} [{ci['p2.5']:.2f}, {ci['p97.5']:.2f}]")
print(f"full-run V2 verdict: {ref2['verdict']} (failed: {ref2['failed']})")

### 6.5 The known-heuristic corpus (`corpus.py`, rebuildable part) and the ab grid

The novelty screen compares every candidate with every known heuristic. The full corpus has 86 entries; this cell
rebuilds the part that needs no external file:

* the textbook rules (FF, BF, WF, AWF) and Sum-of-Squares;
* FunSearch's c12 (vectorised) and c14;
* the LLM4AD inline baselines;
* the published ab members and the two tuned ab members.

Missing are 60 EoH programs, ReEvo, HSEvo and MCTS-AHD, which live in `ext/`. Every member of the run's corpus that
was the nearest neighbour of one of the 11 notable programs is rebuilt here.

Decision vectors are computed through the sandbox, with the probes-only task `corpus.py` uses. The ab grid
(3,843 points in the original) is thinned by `AB_A_STEP` and `AB_B_STEP`. The tuned members are the run's own
picks, argmin F over the first C* points of the committed grid permutation; recomputing them would take 13 minutes.

In [ ]:
# ---- corpus.py (subset; ROOT from the config cell)
CORPUS_DIR = ROOT / "data" / "corpus"
HELDOUT_PRIMARY = ["H-FSW5k", "H-OR3"]
AB_POINTS = [(a, b, base) for a in range(0, 21, AB_A_STEP) for b in range(0, 61, AB_B_STEP)
             for base in ("FF", "BF", "WF")]  # notebook: thinned by the config steps (original range(21) x range(61))
AB_ORDER_SEED = 20261025

SRC_SS = ("def priority(item, bins):\n"
          "    # Sum-of-Squares (Csirik et al. 2006), adapted: N(g) counts the passed OPEN bins with gap g, 0 < g < cap\n"
          "    cap = int(bins.max())\n"
          "    r = bins.astype(int)\n"
          "    counts = np.bincount(r[r < cap], minlength=cap + 1)\n"
          "    delta = np.zeros(len(bins), dtype=float)\n"
          "    is_open = r < cap\n"
          "    delta[is_open] += 1 - 2 * counts[r[is_open]]\n"
          "    new_gap = r - int(item)\n"
          "    pos = new_gap > 0\n"
          "    delta[pos] += 2 * counts[new_gap[pos]] + 1\n"
          "    return -delta\n")
SRC_C12_VEC = ("def priority(item, bins):\n    g = bins - item\n"
               "    return np.select([g <= 2, g <= 3, g <= 5, g <= 7, g <= 9, g <= 12, g <= 15, g <= 18, g <= 20, g <= 21],\n"
               "                     [4, 3, 2, 1, 0.9, 0.95, 0.97, 0.98, 0.98, 0.98], 0.99)\n")


def sha(b: bytes | str) -> str:
    return hashlib.sha256(b.encode() if isinstance(b, str) else b).hexdigest()


def priority_entries() -> tuple[list[dict], dict]:
    E: list[dict] = []
    stats: dict = {}

    def add(name: str, family: str, code: str, note: str, prov: dict | None = None, key: bool = False) -> None:
        E.append({"name": name, "family": family, "code": code, "code_sha256": sha(code), "adapter_note": note,
                  "priority_form": True, "key_baseline": key, "norm_key": sandbox.normalise_key(code),
                  **(prov or {"url": "", "commit": "", "access_utc": utc(), "file_sha256": ""})})

    add("FF", "textbook", bp.SRC_FF, "First Fit (Johnson 1973); bp.SRC_FF", key=True)
    add("BF", "textbook", bp.SRC_BF, "Best Fit; FunSearch's seed priority -(bins - item)", key=True)
    add("WF", "textbook", bp.SRC_WF, "Worst Fit among open bins, new bin if none fits; bp.SRC_WF", key=True)
    add("AWF", "textbook", probes.SRC_AWF, "Almost Worst Fit (Johnson 1974); second-largest gap among open bins", key=True)
    add("SS", "sum_of_squares", SRC_SS, "Sum-of-Squares (Csirik, Johnson, Kenyon, Orlin, Shor, Weber; STOC 2000/JACM "
        "2006) in priority form; adapted: sees only fitting bins", key=True)
    # notebook: c12 / c14 verbatim from the FunSearch notebook, EoH runs 1-3, ReEvo, HSEvo, MCTS-AHD, LLM4AD's
    # template file and the dependency-scan rules read files under ext/ and data/corpus/dep_scan (not shipped)
    add("c12_vec", "funsearch", SRC_C12_VEC, "bp.prio_c12_vec (vectorised c12)")
    add("LLM4AD_tutorial", "llm4ad", "import numpy as np\n\ndef priority(item, bins):\n    return bins - item\n",
        "LLM4AD tutorial template (cell 13) body `return bins - item`")
    add("LLM4AD_eval_default", "llm4ad", "def priority(item, valid_bins):\n    priorities = -valid_bins\n"
        "    return priorities\n", "LLM4AD evaluation.py __main__ default priority (-valid_bins)")
    add("abFF_5_24", "ab", bp.ab_source(5, 24, "FF"), "ab-FirstFit a=5 b=24 (2510.27353 Fig. 9a)", key=True)
    add("abWF_1_21", "ab", bp.ab_source(1, 21, "WF"), "ab-WorstFit a=1 b=21 (2510.27353 Fig. 9b)", key=True)
    add("abBF_1_21", "ab", bp.ab_source(1, 21, "BF"), "ab-BestFit a=1 b=21: unpublished parameters (the paper selects "
        "abBF parameters per instance and prints none)")
    return E, stats


def run_tasks(pool: sandbox.EvalPool, tasks: list[dict]) -> list[dict]:
    futs = [pool.submit(t) for t in tasks]
    out = []
    for i, f in enumerate(futs):
        out.append(f.result())
        if (i + 1) % 250 == 0:
            logger.info(f"  {i + 1}/{len(futs)} done")
    return out


def decision_list(res: dict) -> list[int] | None:
    b = res.get("probes", {}).get("P1")
    return None if b is None else np.frombuffer(b, dtype=np.uint16).astype(int).tolist()


t0 = time.time()
entries, stats = priority_entries()
# the tuned ab members (corpus.py main step 2 adds them by name when they are not already in the corpus)
for cs, t in data["metadata"]["reference"]["tuned_ab"]["prefix_argmin"].items():
    code = bp.ab_source(t["a"], t["b"], t["base"])
    nk = sandbox.normalise_key(code)
    if any(e["norm_key"] == nk for e in entries):
        continue
    entries.append({"name": f"abTUNED_{t['base']}_{t['a']}_{t['b']}", "family": "ab_tuned", "code": code,
                    "code_sha256": sha(code), "priority_form": True, "key_baseline": True, "norm_key": nk,
                    "adapter_note": f"tuned ab member: argmin F over the first C* points of the committed "
                                    f"permutation (rng {AB_ORDER_SEED})", "F": t["F"], "tuned_for_c_star": [int(cs)]})
pool = sandbox.EvalPool(N_WORKERS, ROOT, as_gib=2.0)
# probes-only tasks, as corpus.py runs for entries without decisions
res = run_tasks(pool, [{"code": e["code"], "sets": [], "limit": 60.0, "probes": ["P1"], "probe_limit": 60.0}
                       for e in entries])
for e, r in zip(entries, res):
    e["decisions"] = decision_list(r) if r["error_type"] == "ok" else None
# notebook: c14 from its bp function form (the notebook's verbatim source is not shipped), through probes.decide
p1_arrs = probes.state_arrays(json.loads((ROOT / "data" / "probes" / "P1.json").read_text())["states"])
entries.append({"name": "c14", "family": "funsearch", "code": None, "key_baseline": True,
                "adapter_note": "FunSearch Weibull heuristic as bp.prio_c14",
                "decisions": [probes.decide(bp.prio_c14, it, arr) for it, arr in p1_arrs]})
ab_res = run_tasks(pool, [{"code": bp.ab_source(a, b, base), "sets": [], "limit": 60.0, "probes": ["P1"],
                           "probe_limit": 60.0} for a, b, base in AB_POINTS])
pool.close()
grid = [{"i": i, "a": a, "b": b, "base": base, "error_type": r["error_type"], "F": None,
         "decisions": decision_list(r) if r["error_type"] == "ok" else None}
        for i, ((a, b, base), r) in enumerate(zip(AB_POINTS, ab_res))]
(CORPUS_DIR / "corpus.json").write_text(json.dumps({"utc": utc(), "n_priority_form": len(entries), "entries": entries,
                                                    "stats": stats}, separators=(",", ":")))
(CORPUS_DIR / "ab_grid.json").write_text(json.dumps({"points": grid}, separators=(",", ":")))
# corpus.py step 5: agreement matrix and the known-equivalence calibration pairs
names, CV, ents = corpus_vectors()
agree = cluster.agree_counts(CV) / cluster.N_STATES
idx = {n: i for i, n in enumerate(names)}
print(f"corpus subset: {len(names)} entries with decisions; ab grid: {len(grid)} points ({time.time() - t0:.1f}s)")
print("calibration pairs (the run: c14~abWF_1_21 = 0.59, i.e. the 0.9 rule is not selective there):",
      {p: round(float(agree[idx[p.split('~')[0]], idx[p.split('~')[1]]]), 3)
       for p in ("c14~abWF_1_21", "c12_vec~abFF_5_24", "BF~LLM4AD_eval_default", "FF~BF")})

### 6.6 V6: how often do offspring just rename their parent?

An offspring counts as a *rename* when its decisions agree with a parent's on at least 95 % of states. In round 0
the only parent is best fit. `stage_v6` is verbatim and reads the replay's rows and decision vectors; the corpus
above supplies best fit's vector. In the full run IDEA renamed least (0.44) and T1.0 most (0.86).

In [ ]:
v6_demo = stage_v6()
ref6 = data["metadata"]["reference"]["v6"]
print(f"{'arm':5s} {'valid':>5s} {'renames':>7s} {'rate':>6s}  {'new-idea comment':>16s} | full-run rate")
for a, v in v6_demo["per_arm"].items():
    print(f"{a:5s} {v['n_valid']:5d} {v['renames']:7d} {v['rate']:6.2f}  {v['new_idea_claim_share']:16.2f} | "
          f"{ref6['per_arm_rate'][a]:.2f}")
if "verdict" in v6_demo:
    print(f"V6 rule on the replayed round: {v6_demo['verdict']} (full run: {ref6['verdict']})")

### 6.7 V1: re-scoring the full run's best programs

The replayed round does not beat best fit (see 5.6); in the full run, no arm did within its first 100 calls either.
The programs that beat it were found at rounds 49-78. Each notable program is re-scored from its code:

1. with `corpus.py`'s selection task: F on the 25 selection instances plus the probe decisions;
2. once on the primary held-out sets, with `postsearch.eval_heldout`.

`analysis.v1` then applies the committed rule. One NOVEL program (`main-0-DTB-1671`, also DTB's best) calls
`np.random.rand`, so its scores move a little from run to run.

In [ ]:
NOTABLE = [e for e in EXAMPLES if e["metadata_role"] != ["replay_round0"]][:N_NOTABLE]
fit = bp.Fitness(bp.load_set("SEL-OR"), bp.load_set("SEL-W"), BF_OR_EXC, BF_W_EXC)
t0 = time.time()
pool = sandbox.EvalPool(N_WORKERS, ROOT, as_gib=2.0)
# corpus.py's selection task for known programs: F on SEL-OR + SEL-W and the P1 probes, with 60 s limits instead of
# the search's 10 s (EVAL_TASK), so that a slower CPU cannot time a known program out
rescored = [pool.submit({"code": e["output"], "sets": ["SEL-OR", "SEL-W"], "limit": 60.0, "fail_fast": True,
                         "probes": ["P1"], "probe_limit": 60.0}).result() for e in NOTABLE]
pool.close()
ho = eval_heldout([e["output"] for e in NOTABLE], HELDOUT_PRIMARY)
cands = []
for e, r, h in zip(NOTABLE, rescored, ho):
    b25 = r["bins"]["SEL-OR"] + r["bins"]["SEL-W"] if r["error_type"] == "ok" else None
    dec = np.frombuffer(r["probes"]["P1"], dtype=np.uint16).copy() if r["error_type"] == "ok" else None
    cands.append({"pid": e["metadata_pid"], "stream": f"main:{e['metadata_seed']}_{e['metadata_arm']}",
                  "arm": e["metadata_arm"], "seed": e["metadata_seed"], "run": "main", "round": e["metadata_round"],
                  "call_id": e["metadata_call_id"], "roles": e["metadata_role"], "code": e["output"],
                  "F": fit.F(b25) if b25 else None, "F_recorded": float(e["predict_F"]), "dec": dec,
                  "dec_identical": dec is not None and hashlib.sha256(dec.tobytes()).hexdigest() == e["metadata_dec_sha256"],
                  "exc_FSW5k": h["heldout_excess_pct"].get("H-FSW5k"), "exc_OR3": h["heldout_excess_pct"].get("H-OR3"),
                  "heldout_excess_pct": h["heldout_excess_pct"], "heldout_error": h["error"],
                  "ref": {k[len("metadata_ref_"):]: v for k, v in e.items() if k.startswith("metadata_ref_")}})
print(f"re-scored {len(cands)} programs in {time.time() - t0:.1f}s")
print(f"{'pid':18s} {'role':22s} {'F now':>7s} {'F run':>7s} {'P1 same':>7s} {'FSW5k %':>8s} {'OR3 %':>6s} | run: FSW5k / OR3")
for c in cands:
    rh = c["ref"].get("heldout_excess_pct") or {}
    print(f"{c['pid']:18s} {','.join(c['roles']):22s} {c['F']:7.4f} {c['F_recorded']:7.4f} {str(c['dec_identical']):>7s} "
          f"{c['exc_FSW5k']:8.2f} {c['exc_OR3']:6.2f} | {rh.get('H-FSW5k', float('nan')):.2f} / {rh.get('H-OR3', float('nan')):.2f}")

# V1 with the committed rule: best (by F) program of each arm, held-out excess vs 0.8 x best fit
best = {}
for c in cands:
    if any(r.startswith("best_of_arm:") for r in c["roles"]):
        best[c["stream"]] = {k: c[k] for k in ("pid", "arm", "seed", "run", "F", "code", "call_id", "round",
                                               "heldout_excess_pct", "heldout_error")}
bests = {a: best_of_arm(best, a) for a in ["T1.0", "T1.6", "DTB", "RPB", "IDEA"]}
v1_demo = analysis.v1(bests)
print(f"\nV1 bars: H-FSW5k <= {analysis.THR_FSW5K:.3f} %, H-OR3 <= {analysis.THR_OR3:.3f} % (0.8 x best fit)")
for a, v in v1_demo["per_arm"].items():
    print(f"  {a:5s} pass={v['pass']!s:5s} FSW5k {v['exc_FSW5k'] if v['exc_FSW5k'] is None else round(v['exc_FSW5k'], 2)} "
          f"OR3 {v['exc_OR3'] if v['exc_OR3'] is None else round(v['exc_OR3'], 2)}")
print(f"V1 verdict from these programs: {v1_demo['verdict']} (full run: {data['metadata']['reference']['v1']['verdict']})")

### 6.8 V3: the novelty screen

This is `postsearch.stage_v3`'s logic applied to the notable programs. In the run, the candidates were the 20
best-F behaviour clusters of every stream, 211 in all. A candidate is **NB** (novel-behaviour) when three conditions hold:

* it passes both V1 bars;
* its decisions agree with every corpus entry on fewer than 90 % of states;
* the *strict* version also requires fewer than 90 % against every ab-grid point.

A positive control permutes the corpus vectors; the run's caveat was that this did no work. NB programs are then
grouped into joint behaviour clusters. In the run, an arm-blind manual read classed 30 audited representatives
(shown here as the run's class), and LLM judges were advisory because their gate was unmet.

In [ ]:
def stage_v3(cands: list[dict]) -> dict:
    """postsearch.stage_v3 on the given candidates. notebook: the run's candidates were the top-20 behaviour clusters
    of every stream; here the notable programs, already scored on H-FSW5k / H-OR3 in the cell above."""
    names, CV, ents = corpus_vectors()
    ab = json.loads((ROOT / "data" / "corpus" / "ab_grid.json").read_text())
    ABV = np.array([p["decisions"] for p in ab["points"] if p["decisions"] is not None], dtype=np.uint16)
    ab_names = [f"ab{p['base']}({p['a']},{p['b']})" for p in ab["points"] if p["decisions"] is not None]
    logger.info(f"V3: {len(cands)} candidates")
    Dc = np.stack([c["dec"] for c in cands])
    A = cluster.agree_counts(Dc, CV) / cluster.N_STATES
    A_ab = cluster.agree_counts(Dc, ABV) / cluster.N_STATES
    thr_fs, thr_or = analysis.THR_FSW5K, analysis.THR_OR3
    for c, a, aab in zip(cands, A, A_ab):
        top3 = np.argsort(-a, kind="stable")[:3]
        c.update(max_corpus_agreement=float(a.max()), nearest_corpus=names[int(a.argmax())],
                 nearest3=[[names[int(j)], float(a[j])] for j in top3],
                 max_abgrid_agreement=float(aab.max()), nearest_abgrid=ab_names[int(aab.argmax())])
        c["pass_FSW5k"] = c["exc_FSW5k"] is not None and c["exc_FSW5k"] <= thr_fs
        c["pass_OR3"] = c["exc_OR3"] is not None and c["exc_OR3"] <= thr_or
        c["pass_agree"] = c["max_corpus_agreement"] < 0.9
        c["NB"] = c["pass_FSW5k"] and c["pass_OR3"] and c["pass_agree"]
        c["NB_strict"] = c["NB"] and c["max_abgrid_agreement"] < 0.9
    nb = [c for c in cands if c["NB"]]
    out = {"utc": utc(), "n_candidates": len(cands), "n_NB": len(nb), "n_NB_strict": sum(c["NB_strict"] for c in cands),
           "fail_counts": {"OR3": sum(not c["pass_OR3"] for c in cands), "FSW5k": sum(not c["pass_FSW5k"] for c in cands),
                           "agreement": sum(not c["pass_agree"] for c in cands),
                           "pass_both_heldout": sum(c["pass_OR3"] and c["pass_FSW5k"] for c in cands)}}
    # positive control: permuting each corpus vector across states must raise the NB count
    rng = np.random.default_rng(20261030)
    CVp = np.stack([v[rng.permutation(len(v))] for v in CV])
    Ap = cluster.agree_counts(Dc, CVp) / cluster.N_STATES
    out["positive_control_NB_with_permuted_corpus"] = int(sum(c["pass_FSW5k"] and c["pass_OR3"] and ap.max() < 0.9
                                                              for c, ap in zip(cands, Ap)))
    # joint clusters of the NB set + representatives
    if nb:
        Dn = np.stack([c["dec"] for c in nb])
        jl, nj = cluster.complete_linkage(Dn)
        for j in range(nj):
            mem = [c for c, l in zip(nb, jl) if l == j]
            r = min(mem, key=lambda c: (c["F"], len(c["code"])))
            r["joint_cluster"], r["joint_size"] = j, len(mem)
            r["joint_members"] = [m["pid"] for m in mem]
        out["n_joint_clusters"] = nj
    else:
        out["n_joint_clusters"] = 0
    # notebook: the arm-blind read sheet, the sealed arm map and the 8-set held-out run of the audited representatives
    # (audit/*.json) are not written
    strip = lambda c: {k: v for k, v in c.items() if k != "dec"}  # noqa: E731
    out["candidates"] = [strip(c) for c in cands]
    wjson("v3_candidates.json", out)
    logger.info(f"V3: NB {len(nb)} (strict {out['n_NB_strict']}), joint clusters {out['n_joint_clusters']}; "
                f"positive control NB {out['positive_control_NB_with_permuted_corpus']}")
    return out


v3_demo = stage_v3(cands)
print(f"\n{'pid':18s} {'max agree':>9s} {'nearest corpus':>16s} {'run':>16s} {'max ab':>7s} {'nearest ab':>12s} "
      f"{'NB':>5s} {'strict':>6s}  run class")
for c in cands:
    r = c["ref"]
    print(f"{c['pid']:18s} {c['max_corpus_agreement']:9.4f} {c['nearest_corpus']:>16s} "
          f"{(r.get('nearest_corpus') or '') + ' ' + format(r.get('max_corpus_agreement') or 0, '.4f'):>16s} "
          f"{c['max_abgrid_agreement']:7.3f} {c['nearest_abgrid']:>12s} {c['NB']!s:>5s} {c['NB_strict']!s:>6s}  "
          f"{r.get('final_class', '')}")
ref3 = data["metadata"]["reference"]["v3"]
print(f"\nfull run: {ref3['n_candidates']} candidates, {ref3['n_NB']} NB, {ref3['n_joint_clusters']} joint clusters, "
      f"{ref3['n_audited']} audited, {ref3['n_novel']} NOVEL -> V3 {ref3['verdict']} ({ref3['flag']})")
novel = sorted([c for c in cands if c["ref"].get("final_class") == "NOVEL"], key=lambda c: len(c["code"]))
if novel:
    print(f"\nThe NOVEL family, shortest member ({novel[0]['pid']}). The manual read: {novel[0]['ref']['rule_in_words']}\n")
    print(novel[0]["code"])

## 7. Results

**The tables** set this run beside the full run, per arm: validity, behaviour clusters and the rename rate of the
replayed round, then each arm's best program on the held-out sets.

**The figure** has three panels:

* **(a) Behaviour space, after `figures.py` F4.** Classical MDS of `1 - decision agreement` places every replayed
  offspring (dot size = number of calls with that exact behaviour), the arm bests, the NOVEL programs, the corpus
  entries and the ab grid in one 2-D map. The round-0 offspring pile up on best fit. The programs that beat best fit
  sit in another region, near the ab family but at below 90 % agreement with any member.
* **(b) V1.** Held-out excess of each arm's best program against best fit and the 0.8 x best-fit bars.
* **(c) The DTB operator.** The entropy at the forced position, for DTB (always the most uncertain eligible token)
  and its random-position placebo RPB.

In [ ]:
from matplotlib.patches import Patch  # notebook: legend patches

# ---- figures.py (verbatim): arm colours and classical MDS
COL = {"T1.0": "#1f77b4", "T1.6": "#ff7f0e", "DTB": "#d62728", "RPB": "#9467bd", "IDEA": "#2ca02c", "NS": "#7f7f7f",
       "corpus": "#000000"}


def classical_mds(Dist: np.ndarray, k: int = 2) -> tuple[np.ndarray, float]:
    n = len(Dist)
    J = np.eye(n) - np.ones((n, n)) / n
    Bm = -0.5 * J @ (Dist ** 2) @ J
    w, V = np.linalg.eigh(Bm)
    idx = np.argsort(w)[::-1][:k]
    X = V[:, idx] * np.sqrt(np.maximum(w[idx], 0))
    dhat = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))
    iu = np.triu_indices(n, 1)
    stress = float(np.sqrt(((Dist[iu] - dhat[iu]) ** 2).sum() / (Dist[iu] ** 2).sum()))
    return X, stress


# ---- per-arm summary table
ref = data["metadata"]["reference"]
print(f"Replayed round 0 (seed 0, {CALLS_PER_ROUND} calls per arm) vs the full run (2 seeds x {ref['v2']['C_s']} calls per arm)")
print(f"{'arm':5s} | {'valid':>5s} {'clusters':>8s} {'renames':>7s} | {'full: clusters':>14s} {'renames':>7s} | "
      f"{'best: FSW5k %':>13s} {'OR3 %':>6s} {'(run)':>13s}")
for a in ARMS_MAIN:
    s0 = cl["streams"][f"main:0_{a}"]
    b = bests.get(a) or {}
    rv1 = ref["v1"]["per_arm"].get(a) or ref["v1"]["beside"].get(a) or {}
    fs, o3 = b.get("exc_FSW5k"), b.get("exc_OR3")
    print(f"{a:5s} | {s0['n_valid']:5d} {s0['D']:8d} {v6_demo['per_arm'][a]['rate']:7.2f} | {full_D[a]:14d} "
          f"{ref['v6']['per_arm_rate'][a]:7.2f} | {fs if fs is None else round(fs, 2):>13} {o3 if o3 is None else round(o3, 2):>6} "
          f"{format(rv1.get('exc_FSW5k', float('nan')), '.2f') + ' / ' + format(rv1.get('exc_OR3', float('nan')), '.2f'):>13s}")
bf_ref = ref["v1"]["beside"]["BF"]
tuned = ref["v1"]["beside"]["tuned_ab"]
print(f"best fit: {bf_ref['H-FSW5k']:.2f} / {bf_ref['H-OR3']:.2f} %   tuned ab {tuned['name']}: {tuned['H-FSW5k']:.2f} / "
      f"{tuned['H-OR3']:.2f} %   verdicts (full run): {data['metadata']['reference']['v1']['verdict']} V1, "
      f"{ref['v2']['verdict']} V2, {ref['v3']['verdict']} V3, {ref['v6']['verdict']} V6")

# ---- (a) behaviour space: replayed offspring (unique behaviours), notable programs, corpus, ab grid
pts = []  # (kind, arm, multiplicity, decision vector, pid)
for key, s in cl["streams"].items():
    rows_s, D = load_stream(s["run"], s["arm"], s["seed"], s["calls"])
    vm = np.array([analysis.valid(r) and not r.get("probe_failed") for r in rows_s], dtype=bool)
    if vm.any():
        uniq, cnt = np.unique(D[vm], axis=0, return_counts=True)
        pts += [("replay", s["arm"], int(c), u, None) for u, c in zip(uniq, cnt)]
pts += [("novel" if c["ref"].get("final_class") == "NOVEL" else "best", c["arm"], 1, c["dec"], c["pid"]) for c in cands]
names, CV, ents = corpus_vectors()
abg = json.loads((ROOT / "data" / "corpus" / "ab_grid.json").read_text())["points"]
ABV = np.array([p["decisions"] for p in abg if p["decisions"] is not None], dtype=np.uint16)
ABV = ABV[::max(1, len(ABV) // 1000)]  # notebook: at most ~1,000 grid points in the map (MDS cost grows as n^2)
V = np.concatenate([np.stack([p[3] for p in pts]), CV, ABV]).astype(np.uint16)
C = cluster.agree_counts(V)
Dist = 1.0 - C / cluster.N_STATES
np.fill_diagonal(Dist, 0.0)
Dist = np.maximum(Dist, Dist.T)
X, stress = classical_mds(Dist)
n_p, n_c = len(pts), len(CV)

fig, axes = plt.subplots(1, 3, figsize=(19, 6.2), gridspec_kw={"width_ratios": [1.45, 1.0, 0.75]})
ax = axes[0]
ax.scatter(X[n_p + n_c:, 0], X[n_p + n_c:, 1], s=6, color="#c8c8c8", label=f"ab grid ({len(ABV)} pts)", zorder=1)
for arm in ARMS_MAIN:
    m = [i for i, p in enumerate(pts) if p[0] == "replay" and p[1] == arm]
    if m:
        ax.scatter(X[m, 0], X[m, 1], s=[18 + 14 * pts[i][2] for i in m], color=COL[arm], alpha=0.55,
                   label=f"{arm} round 0 ({sum(pts[i][2] for i in m)} valid)", zorder=3)
mb = [i for i, p in enumerate(pts) if p[0] == "best"]
mn = [i for i, p in enumerate(pts) if p[0] == "novel"]
if mb:
    ax.scatter(X[mb, 0], X[mb, 1], s=90, marker="D", c=[COL[pts[i][1]] for i in mb], edgecolor="k",
               label="arm best (KNOWN family)", zorder=5)
if mn:
    ax.scatter(X[mn, 0], X[mn, 1], s=230, marker="*", color="gold", edgecolor="k", label="NOVEL", zorder=6)
ax.scatter(X[n_p:n_p + n_c, 0], X[n_p:n_p + n_c, 1], s=40, marker="x", color="k", label=f"corpus ({n_c})", zorder=4)
for j, e in enumerate(ents):  # figures.f4: label the key entries
    if e["name"] in ("BF", "FF", "WF", "AWF", "SS", "c12_vec", "c14", "abFF_5_24", "abWF_1_21") or \
            e["family"] == "ab_tuned":
        ax.annotate(e["name"], X[n_p + j], fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("MDS dimension 1 (behaviour space: 1 - decision agreement)")
ax.set_ylabel("MDS dimension 2")
ax.set_title(f"(a) Behaviour space on 2,000 probe states, classical MDS (stress-1 = {stress:.3f})", fontsize=9)
ax.legend(fontsize=7, loc="best")

# ---- (b) V1: held-out excess of each arm's best program vs best fit and the 0.8 x BF bars
ax = axes[1]
arms_b = [a for a in ["T1.0", "T1.6", "DTB", "RPB", "IDEA"] if bests.get(a)]
xs = np.arange(len(arms_b) + 1)  # + the run's tuned ab baseline (reference values)
cols = [COL[a] for a in arms_b] + ["#7f7f7f"]
ax.bar(xs - 0.2, [bests[a]["exc_FSW5k"] for a in arms_b] + [tuned["H-FSW5k"]], 0.4, color=cols)
ax.bar(xs + 0.2, [bests[a]["exc_OR3"] for a in arms_b] + [tuned["H-OR3"]], 0.4, color=cols, alpha=0.45, hatch="//")
hl = [ax.axhline(y, ls=ls, lw=1, color=c, label=lab) for y, ls, c, lab in (
    (analysis.BF_FSW5K, "-", "k", "best fit, H-FSW5k"), (analysis.THR_FSW5K, "--", "k", "V1 bar H-FSW5k (0.8 x BF)"),
    (analysis.BF_OR3, "-", "r", "best fit, H-OR3"), (analysis.THR_OR3, "--", "r", "V1 bar H-OR3 (0.8 x BF)"))]
ax.set_xticks(xs)
ax.set_xticklabels(arms_b + [tuned["name"].replace("abTUNED_", "tuned ab\n")])
ax.set_ylabel("excess over the L1 bound (%), lower is better")
ax.set_title("(b) V1: best program per arm, re-scored here (grey: tuned ab, run value)", fontsize=9)
ax.legend(handles=[Patch(facecolor="0.35", label="H-FSW5k (FunSearch Weibull 5k)"),
                   Patch(facecolor="0.8", hatch="//", label="H-OR3 (OR-Library binpack3)")] + hl, fontsize=6.5, loc="upper left")

# ---- (c) DTB vs RPB: entropy of the forced position and the most uncertain eligible position
ax = axes[2]
for k, arm in enumerate(["DTB", "RPB"]):
    fr = [r["fork"] for r in fork_rows if r["arm"] == arm]
    if fr:
        jit = np.linspace(-0.12, 0.12, len(fr))
        ax.scatter(k + jit, [f["H_j"] for f in fr], color=COL[arm], s=28, label=f"{arm}: H at the forced token")
        ax.scatter(k + jit, [max(f["H"][t] for t in f["E"]) for f in fr], marker="_", s=160, color="k",
                   label="max H over eligible tokens" if k == 0 else None)
ax.set_xticks([0, 1])
ax.set_xticklabels(["DTB", "RPB"])
ax.set_xlim(-0.6, 1.6)
ax.set_ylabel("top-20 entropy at the fork (nats)")
ax.set_title("(c) Where the operator forks (replayed calls)", fontsize=9)
ax.legend(fontsize=6.5, loc="lower left")
fig.tight_layout()
plt.show()